# Supply Chain System — Checkpoint Summary

## What We Built

A logistics layer for a hex-grid strategy game where **food is the fundamental constraint**. Pieces don't just fight — they eat, starve, and die. Every military action requires a working supply chain behind it.

### Core Mechanics

- **Pawns** harvest food from terrain tiles (yield = `harvest_strength × food_tier`)
- **Bishops** carry food along routes between settlements and consumers
- **Queens, Knights, Rooks** consume food each turn — they starve without supply lines
- **Food tiers** are terrain-driven: coastal plains are fertile, mountains are barren
- Pieces have `diet`, `food_capacity`, and `harvest_strength` configured via `FoodProfile`

### The Five Rules

1. Every piece eats `diet` food per turn
2. Shortfall × 5 = starvation damage per turn
3. Pawns harvest `harvest_strength × tile_tier` food when executing HARVEST
4. GIVE transfers surplus food to visible friendly pieces
5. Bishops move along planned routes; their effective delivery = `capacity - (2 × travel_turns × diet)`

### Key Classes

**`FoodProfile`** — per-piece-type food configuration (diet, capacity, harvest_strength).
Applied via `FoodProfile.apply(piece)` in `Piece.__post_init__`.

**`FoodSimulator`** — runs the turn loop (eat → starve → harvest → give → move),
records events per piece, renders `show(sim)` as alive/health/food line charts.

**`SupplyPlanner`** — the math engine for supply logistics:
- `net_delivery(path_cost)` — food remaining after a bishop eats its own cargo
- `throughput(cost, n_bishops)` — sustained food/turn at a given distance
- `bishops_needed(cost, diet)` — minimum couriers to sustain a consumer
- `max_range()` — furthest a bishop can go and still deliver food
- `supply_range_map()` — Dijkstra reachability from settlement
- `find_target()` — locate elevated, reachable positions for consumers
- `create_supply_line()` — place and route bishops automatically
- `provision_consumer()` — set starting food based on delivery wait time
- `show(planner)` — throughput curves + bishops-needed heatmap

**`ColonyOptimizer`** — MILP-based resource allocation:
- Given N consumers at various distances with priorities (queen=10, knight=3, etc.)
- Allocates limited bishops and harvest budget to maximize total priority fed
- Constraints: total bishops, throughput ≥ diet, harvest ≥ total demand
- `show(optimizer)` — bishop allocation bars, throughput vs diet, harvest budget pie

### Key Insight: The Bishop Fuel Problem

A bishop carrying food eats its own cargo in transit. The **net delivery** formula:



### Making the game
This is a great design question. The answer I'd push toward: the player decides where and what priority, the system shows them what it costs, and they confirm.

The decision flow:

1. Explore — Player sees a supply range heatmap overlaid on the board. Color encodes "how expensive is it to sustain a piece here" — green (1 bishop), yellow (2), orange (3), red (unreachable). This is supply_range_map() rendered on the hex grid. The player immediately sees the shape of their logistical reach.

2. Designate — Player clicks hexes to mark them as expansion targets. Maybe they want a knight outpost on that hilltop, or the queen pushed forward to a defensible position. Each click adds a pin to the map.

3. Preview — System runs ColonyOptimizer and shows the result before committing. "You want 3 targets? Here's the plan: queen gets 2 bishops, knight gets 1, rook is ❌ too far — we'd need 2 more pawns." The player sees the harvest budget pie and the tradeoffs before any pieces move.

4. Commit or adjust — Player can reprioritize (drag the rook closer, drop a target, add pawns). Each change re-runs the optimizer and updates the preview. When they're happy, they commit and the bishops get their marching orders.

5. Micromanage (optional) — The supply lines run on autopilot via patrol instructions. But the player can manually reroute a bishop, override a patrol, pull a supply line to react to threats. This is the "cool and unique" part — you don't have to touch it, but you can.

For the AI opponent, the same code runs but steps 2-4 collapse into:
Copied!

# AI picks targets using its own strategy (threat assessment, territory, etc.)
targets = ai.pick_expansion_targets(board, planner)

# Same optimizer the player uses
consumers = opt.analyze_consumers(targets)
result = opt.optimize(consumers)

# Execute directly
for a in result['assignments']:
    if a['fed']:
        planner.create_supply_line(a['consumer'].hex_idx, board,
                                   n_bishops=a['bishops'])

The AI uses the exact same logistics constraints — no cheating on food. It just skips the UI.

What the player doesn't decide (and shouldn't have to):

    Exact number of bishops per route (optimizer)
    Patrol route instructions (pathfinder)
    Starting food amounts (provisioner)
    When to harvest vs give (patrol loop)

What the player does decide (the fun part):

    Where to expand
    What to prioritize when resources are scarce
    When to cut losses and abandon a position
    Whether to micromanage a critical supply line

The heatmap is the key design insight — it turns complex math into spatial intuition. A player looks at it and thinks "I can reach there cheaply but there is expensive" without ever seeing a number. That's good game design.

### Visualizing where to go

```python
# Just the supply heatmap on cream
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement),
    terrain=myTerr,
)

# With rivers too
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='bishops_needed'),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)

# Compare different fields
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='net_delivery'),
    terrain=myTerr,
)

# Reuse an existing planner (avoids recomputing)
planner = SupplyPlanner(myTerr, settlement=best_settlement)
planner.compute_supply_fields()

TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, planner=planner),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)
```


Good thinking. Here's where I'd draw the line between "build now" and "build later":

### Flesh out now (supply-only)

**1. `_reoptimize` implementation** — this is pure supply math, no combat needed:

```python
@patch
def _reoptimize(self: BattlePlan):
    """Re-run optimizer respecting player overrides."""
    opt = self.optimizer
    
    # Filter out vetoed consumers
    active = [a['consumer'] for a in self.assignments
              if a['consumer'].hex_idx not in self.vetoed]
    
    # Apply locked constraints — force f_i = 1 for locked consumers
    # Apply boosted — set minimum bishops per route
    result = opt.optimize(active,
                          locked=self.locked,
                          min_bishops=self.boosted)
    
    self.assignments = result['assignments']
    self.harvest_budget = result['harvest_available']
    return self
```

**2. `__ft__` for BattlePlan** — the interactive preview card. This is what makes it fun:

```python
@patch
def __ft__(self: BattlePlan):
    """Interactive plan preview with veto/lock/boost controls."""
    rows = []
    for a in sorted(self.assignments,
                    key=lambda x: -x['consumer'].priority):
        c = a['consumer']
        name = getattr(c.piece, 'name', f'hex {c.hex_idx}')
        status = "✅" if a['fed'] else "❌"
        vetoed = c.hex_idx in self.vetoed
        locked = c.hex_idx in self.locked
        
        icon = "🚫" if vetoed else ("🔒" if locked else status)
        
        rows.append(Tr(
            Td(icon),
            Td(Strong(name)),
            Td(f"{c.piece.piece_type.name}"),
            Td(f"{c.diet:.1f}"),
            Td(f"{c.path_cost:.1f}"),
            Td(f"{a['bishops']}"),
            Td(f"{a['throughput']:.2f}"),
        ))
    
    table = Table(
        Thead(Tr(*[Th(h) for h in
            ["", "Name", "Type", "Diet", "Cost", "Bishops", "Throughput"]])),
        Tbody(*rows),
        cls="text-sm"
    )
    
    fed = sum(1 for a in self.assignments if a['fed'])
    total = len(self.assignments)
    
    return Div(
        P(f"🗺 Battle Plan — {fed}/{total} fed, "
          f"{sum(a['bishops'] for a in self.assignments)} bishops committed",
          cls="font-bold"),
        table,
        P(f"🌾 Harvest budget: {self.harvest_budget:.0f} food/turn",
          cls="text-xs opacity-60"),
        cls="space-y-2"
    )
```

**3. King protection priority** — this is just a number tweak, no combat:

```python
# King gets infinite-ish priority — optimizer will NEVER cut the king's supply
PieceType.KING: 20   # already highest, but could be 100 to guarantee it
```

### Think about now, build later

The piece roles you described create a beautiful **rock-paper-scissors** with supply:

| Piece | Speed | Diet | Combat Role | Supply Role |
|-------|-------|------|-------------|-------------|
| **King** | Slow | 4.0 | Must survive | Consumes heavily |
| **Queen** | Medium | 3.0 | Power piece | Consumes heavily |
| **Rook** | Slow | 2.0 | Defends supply lines | Static guard |
| **Knight** | Fast | 1.5 | Raids enemy supply | Light footprint |
| **Bishop** | Medium | 1.5 | None | Courier |
| **Pawn** | Slow | 0.5 | None | Harvester |

The strategic loop becomes:
- **Knights raid enemy supply lines** (fast, cheap to sustain, can hit and run)
- **Rooks defend your supply lines** (slow but tough, stationed along routes)
- **Queen projects power** but needs 2-3 bishops dedicated to her
- **King stays near the settlement** (expensive to supply far away, losing him = game over)

The fun tension: **every rook you place on defense is a rook not attacking, and every knight raiding is a knight not defending.** The supply chain makes this tradeoff real — you can't afford to deploy everything everywhere.


```python

# King gets infinite-ish priority — optimizer will NEVER cut the king's supply
PieceType.KING: 20   # already highest, but could be 100 to guarantee it


# Full battle view: terrain + supply range + routes + piece movements
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement),
    BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)

# Just the piece movements without supply heatmap
TerrainDisplay(
    CreamOverlay(),
    BattlePieceOverlay(battle_plan, num_turns=8, show_pawns=True),
    terrain=myTerr,
)

# Bishops only — see the supply runs
TerrainDisplay(
    CreamOverlay(),
    BattlePieceOverlay(battle_plan, show_consumers=False, show_pawns=False),
    terrain=myTerr,
)
```

In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition, field_of_view

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc

import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq
import io

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds
from HexMagic.water.soil import SoilSystem
from enum import Enum

In [ ]:
myTerr = TerraDemo().japan_korea_map()
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

In [ ]:
basins = DrainageBasins(myTerr)

In [ ]:
#TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

## FoodYield

In [ ]:
#| export


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
#| export
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
# Make sure soil exists

soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
#show(fy)
fy.summary()

can you fix the __ft__ so that it uses tick_labels

I think this is ok. there is a bunch of mountains and oceans

In [ ]:
fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()

Thoughts

lets build the overlay. I know we need terrainPatterns for the dots. What would be a good color to use. I am assuming we will use it either with a cream backgroun or the elevations. does it make sense to have different colors or just different densities

In [ ]:
#| export


def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


In [ ]:
svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


## FoodOverlay

In [ ]:
#| export
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
#!cat ../../HexMagic/game/piece.py

So we now have the basics to have food on the map we should work out the mechanics of how pieces move to food harvest and share food with one another. 

1. we need a dystyra path to pick a target
2. something that will assign a piece
3. the share mechanism. my idea for this is the same a the context where they have a cone coming out from them a sight modified number of hexes. feeding priority is given by rank and then who has the least amount of food. you will always keep 4 turns of your diet of food so you won't starve yourself.

Other things we should add?

In [ ]:
??Piece

So we have food, diet and food_capacity on pieces. If starving you start to loose health. all health lost you die

Diet-proportional

Lets do 5, I am a generous king.

harvest_strength × tile_tier

In [ ]:
#!cat ../../HexMagic/plot/*.py

So we have quite a bit of math that uses hexpostions to find adjacent hexes

allowed_rotations=1

 KING > QUEEN > ROOK > BISHOP > KNIGHT > PAWN

In theory each piece does its thing in isolation. We are going to build rules a level up, but at the piece level if told to give you give.

How much of this can you build?

lets extract out a pieces in sight with a multipler (pieces at higher elevations are going to see farther, but we can figure out by home much by later). it gives us a nice board balance where the higher ground has strategic advantage, but the lower ground has more food.

We now have
```
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance."""
    results = []
    for radius in range(1, max_distance + 1):
        for hex_pos in origin.ring(radius):
            if hex_in_cone(hex_pos - origin, facing, 1):
                results.append(hex_pos)
    return results
```

In [ ]:
#| export
from HexMagic.plot.cube import HexPosition, field_of_view

STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

# Lower index = higher priority (fed first)
RANK_ORDER = {
    PieceType.KING:   0,
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,
}


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)



def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)


In [ ]:
#| export
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    squads: list[Squad],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Squad]:
    """Find pieces visible from piece's location, grouped by squad.
    
    Accepts squads, returns squads containing only visible members.
    Empty squads (no visible members) are dropped.
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    origin = grid.index_to_hexposition(piece.location)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    result = []
    for squad in squads:
        visible_members = []
        for other in squad.alive:
            if other is piece or other.location is None:
                continue
            if allied_only and other.owner_id != piece.owner_id:
                continue

            relative = grid.index_to_hexposition(other.location, piece.location)

            if facing_only:
                if relative not in fov_set:
                    continue
            else:
                if abs(relative) > effective_sight:
                    continue

            visible_members.append(other)

        if visible_members:
            result.append(Squad(id=squad.id, name=squad.name, pieces=visible_members))

    return result


In [ ]:
??Squad

In [ ]:
directions_in_cone??

can pieces_in_ sight take and give back squads

In [ ]:
#| export
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         battlefield: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone.
    
    battlefield is ALL pieces (across all squads) — give targets
    any allied piece in sight, not just squad-mates.
    """
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            grid, elevations, battlefield,
            facing_only=True, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount


## Squad extensions

In [ ]:
#| export
@patch
def __ft__(self: Squad, grid: HexGrid = None, list_pieces: bool = False):
    """Compact squad dashboard — box plots by piece type, inline SVG."""
    alive = self.alive
    all_pieces = self.pieces

    if not all_pieces:
        return Div(P(f"Squad '{self.name}' — empty", cls="text-sm opacity-50"),
                   id=f"squad-{self.id}")

    # --- Group by piece type (ranked order) ---
    types_present = sorted(set(p.piece_type for p in all_pieces),
                           key=lambda t: RANK_ORDER.get(t, 99))
    type_labels = [t.name.title() for t in types_present]

    health_data, food_data, turns_data, distance_data = [], [], [], []
    alive_counts, total_counts = [], []

    # --- Centroid for distance (if grid) ---
    centroid = None
    if grid is not None:
        locs = [p.location for p in alive
                if p.location is not None and p.location >= 0]
        if len(locs) >= 2:
            positions = [grid.index_to_hexposition(loc) for loc in locs]
            cq = sum(p.q for p in positions) / len(positions)
            cr = sum(p.r for p in positions) / len(positions)
            cs = sum(p.s for p in positions) / len(positions)
            centroid = (cq, cr, cs)

    # --- Collect per-type data ---
    for pt in types_present:
        members = [p for p in all_pieces if p.piece_type == pt]
        alive_m = [p for p in members if p.health > 0]
        alive_counts.append(len(alive_m))
        total_counts.append(len(members))

        health_data.append([p.health for p in members])
        food_data.append([p.food for p in members])
        turns_data.append([p.food / max(p.diet, 0.01) for p in members])

        if centroid is not None:
            dists = []
            for p in members:
                if p.location is not None and p.location >= 0:
                    pos = grid.index_to_hexposition(p.location)
                    d = (abs(pos.q - centroid[0])
                       + abs(pos.r - centroid[1])
                       + abs(pos.s - centroid[2])) / 2
                    dists.append(d)
                else:
                    dists.append(float('nan'))
            distance_data.append(dists)

    # --- Build figure ---
    has_dist = centroid is not None and distance_data
    n_panels = 4 if has_dist else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 3.0))
    if n_panels == 1: axes = [axes]

    panels = [
        ("Health",           health_data,   "#e74c3c"),
        ("Food",             food_data,     "#27ae60"),
        ("Food (turns left)", turns_data,   "#f39c12"),
    ]
    if has_dist:
        panels.append(("Distance from center", distance_data, "#3498db"))

    for ax, (title, data, color) in zip(axes, panels):
        if not data or all(len(d) == 0 for d in data):
            ax.set_visible(False)
            continue

        bp = ax.boxplot(data, tick_labels=type_labels, patch_artist=True,
                        medianprops=dict(color='#222', linewidth=1.5),
                        whiskerprops=dict(color='#888'),
                        capprops=dict(color='#888'))
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.55)

        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(axis='y', alpha=0.3, linewidth=0.5)

        # Alive/total labels above each box
        ymax = ax.get_ylim()[1]
        for i, (a, t) in enumerate(zip(alive_counts, total_counts)):
            ax.text(i + 1, ymax * 1.02, f"{a}/{t}",
                    ha='center', va='bottom', fontsize=7, color='#555',
                    fontstyle='italic')

    squad_label = self.name or f"Squad #{self.id}"
    fig.suptitle(squad_label, fontsize=12, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    chart_svg = NotStr(buf.getvalue())

    # --- Summary line ---
    total_alive = sum(alive_counts)
    total_all = sum(total_counts)
    avg_hp = np.mean([p.health for p in alive]) if alive else 0
    min_hp = min((p.health for p in alive), default=0)
    avg_turns = np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0

    summary = P(
        f"🗡 {total_alive}/{total_all} alive · "
        f"❤ avg {avg_hp:.0f} (min {min_hp:.0f}) · "
        f"🌾 {avg_turns:.1f} avg food turns",
        cls="text-xs opacity-60 text-center mt-1"
    )

    parts = [chart_svg, summary]

    if list_pieces:
        parts.append(Divider())
        parts.append(PieceList(all_pieces))

    return Div(*parts, id=f"squad-{self.id}", cls="space-y-2")


## TroopPath

In [ ]:
#| export
@dataclass
class TroopPath:
    """A resolved hex route with cached metadata."""
    hexes: list[int]
    cost: float = 0.0          # total movement cost (sum of _move_cost steps)
    
    def __len__(self):   return len(self.hexes)
    def __iter__(self):  return iter(self.hexes)
    def __bool__(self):  return len(self.hexes) > 1   # empty or single-hex = falsy
    def __getitem__(self, i): return self.hexes[i]
    
    @cached_property
    def as_set(self) -> set[int]:
        return set(self.hexes)
    
    @property
    def start(self) -> int: return self.hexes[0]
    
    @property
    def end(self) -> int:   return self.hexes[-1]
    
    def to_rules(self, grid: HexGrid, start_facing: int = 0) -> list[int]:
        return InstructionList.path_to_rules(self.hexes, grid, start_facing=start_facing)


In [ ]:
#| export
@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None) -> TroopPath:
    """Dijkstra from piece.location to target. Returns TroopPath (falsy if unreachable)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current == target:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)





In [ ]:
#| export
@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier. Returns TroopPath (falsy if none)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)



In [ ]:
#| export

@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Thin wrapper: path to target → InstructionList."""
    path = self.pathfind(target, grid, elevations, countries)
    if not path:
        return InstructionList()
    return InstructionList(path.to_rules(grid, self.facing), cursor=0, patrol=False)

I added it. The feed turn is more complex since you will want to feed things outside of your squad (so you want a sight that goes outside your squad). but lets keep refactoring

I love Hungarians, I was once taught by Egon Balas. I think we want two option - one facing inward as if they are feeding the queen and one facing outward so they could pretect her.

In [ ]:
#| export
class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6



In [ ]:
#| export

@patch
def surround(self: Squad, target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign squad members to surround positions using Hungarian matching.
    
    Args:
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    pieces = self.alive
    if not pieces:
        return {}

    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))

    if not candidates:
        return {}

    n_pieces = len(pieces)
    n_positions = len(candidates)

    # 2. Build cost matrix, caching TroopPaths to avoid double-pathfind
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    paths: dict[tuple[int, int], TroopPath] = {}

    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        eff_sight = piece.effective_sight(elevations, elevation_mult)

        for j, (pos_idx, ring_dist) in enumerate(candidates):
            if ring_dist > eff_sight:
                continue
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue
            paths[(i, j)] = path
            cost[i, j] = path.cost

    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)

    # 4. Generate instruction lists from cached paths
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue

        piece = pieces[i]
        path = paths[(i, j)]
        pos_idx = candidates[j][0]

        rules = path.to_rules(grid, piece.facing)

        # Desired final facing
        if mode == SurroundMode.FEED:
            desired = _facing_toward(grid, pos_idx, target)
        else:
            desired = _facing_away(grid, pos_idx, target)

        # Facing after the last move step
        end_facing = _facing_toward(grid, path[-2], path[-1])

        # Shortest rotation to desired facing
        diff = (desired - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)

    return assignments


In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

# --- Group into a squad ---
escort = Squad(id=1, name="Queen's Escort", pieces=feeders)

print(f"Queen at hex {mid}, squad '{escort.name}' has {len(escort)} members")

# --- Surround with FEED mode ---
orders = escort.surround(queen.location, grid, elevs,
                         mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)}/{len(escort)} pieces to surround positions")
for p in escort.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
#board.render(show_plan=True, num_turns=8)
#show(board)


How about with a pieces overlay and the queen not sitting in water?

# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Find nearest land hex to center for queen
mid = grid.middle
queen_idx = next(
    idx
    for dist in range(10)
    for hp in ([HexPosition.origin()] if dist == 0 else HexPosition.origin().ring(dist))
    for idx in [grid.hexposition_to_index(hp, mid)]
    if idx >= 0 and idx not in grid.invalidRegion and elevs[idx] > 0
)
print(f"Queen placed at hex {queen_idx} (elev={elevs[queen_idx]:.0f})")

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0

# Scatter feeders — skip any that land on water
offsets = [(-3,-2), (-4,1), (2,-3), (3,2), (5,0), (-2,4), (1,-5), (-5,-1)]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, queen_idx)
    if idx < 0 or idx in grid.invalidRegion or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

print(f"{len(feeders)} feeders placed")

# --- Group into squad and surround ---
escort = Squad(id=1, name="Queen's Escort", pieces=feeders)
orders = escort.surround(queen.location, grid, elevs, mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)}/{len(escort)} pieces")
for p in escort.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render with plan overlay ---
PieceBoardPlan(board, num_turns=8)


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


#| export
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#| export
@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()



In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

I can't really see the bars. Are they under? should we use a different color?

so they should extend further towards the edge of the hex we also want thing rotated to face the queen

In [ ]:
#| export
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

So I think we are ready for our sightOverlay. we want to see which hexes are covered by a list of pieces. perhaps the overlay could be a dotted one similar to food, but with the kingdom color

In [ ]:
#| export
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)





@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


 think we need to work on field of view which seems to go wider that 120. in ring 1 it is fine. but in ring 2 it should go straing out from its edges but it bends

These wound up too suttble. I wonder if darkPrimary or baseComp or perhaps larger dots makes more sense.

In [ ]:
#| export
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


In [ ]:
#| export
@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()

Lets do a demo with one piece so I can see the cone

# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

In [ ]:
#| export
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Walks the cone arc per ring — no per-hex tests needed.
    """
    left_edge = facing.rotate(1)     # start of arc (left side)
    step1 = facing.rotate(-1)        # walk from left toward center
    step2 = facing.rotate(-2)        # walk from center toward right
    
    results = []
    for r in range(1, max_distance + 1):
        current = origin + r * left_edge
        for _ in range(r):
            results.append(current)
            current = current + step1
        for _ in range(r):
            results.append(current)
            current = current + step2
        results.append(current)      # final right-edge hex
    return results


# --- Fresh board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid  = board.grid
elevs = board.elevations

# Find a nice land hex near center
mid = grid.middle
scout_idx = None
for dist in range(0, 6):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            scout_idx = idx
            break
    if scout_idx is not None:
        break

# Single scout, facing direction 3 (NE) so cone points up-right
scout = board.add_piece(scout_idx, PieceType.KNIGHT, facing=3, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
scout.sight = 4   # decent range

print(f"Scout at hex {scout_idx} (elev={elevs[scout_idx]:.0f}), facing=3 (NE)")
print(f"Effective sight: {scout.effective_sight(elevs):.2f}")

# Cone only — shows the 120° wedge
show(NotStr(board.render_with_sight(
    [scout],
    num_turns=1,
    facing_only=True   # ← cone mode
)))


So I am wondering about another matching algorith. we want to cover a path so that as a piece walks along it they would be fed. so we need to put pieces on the best yields "outside of the path". and then use a similar bipartite graph to match them. I think these field of view algorithms will come in handy for getting covverage.

def supply_line(path: list[int], feeders: Squad,
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Wrap in Squad and run supply line ---
feeder_squad = Squad(id=1, name="Supply Corps", pieces=feeders)

orders = supply_line(march_path.hexes, feeder_squad, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))


## Food simulator

Now it is time to build our first simulator. this will do moves and food. we will do combat in a different notebook, but we should see about self substaning colonies and even ones that are expanding.

The simulation goes move by move. if a piece does not have any move turns it is excluded.
we order first by rank, then by age oldest first, then by health healthiest first, and then random.
we do moves if a space is occupied then you don't move into it. if the space is invalid you rotate until you are pointed in a good direction.
we then do all gathers.
we then do shares in the same order except instead of health, by least food.
we then compute diet - food is eaten and starving health adjusted. This game does not have rest concept. So once health is lost it can't be regained.

Any thoughts on the food simulator?

Collision on move — two pieces want the same hex this turn. Since you process by rank/age/health, the first mover wins. the second piece just stay put and does not incerement its move counter. That way it still can go when the path becomes available

rotate until valid - we should to a map level check so that we don't have single hex islands.
yes do after each giver distibution and each giver needs 4 cached levels of its own diet.

history - absolutely. much as we have a planning diagram we are going to have resolve diagrams so everyone can see what happened.

I think the simulator, but we should also start thinking about the _FT_ of the squad so we can measure group health, food, how scattered. this way we can itterate and just check the _FT_ to see what is happening.

In [ ]:
??Piece.__ft__

In [ ]:
SVGBuilder.__ft__??

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

actually ft stands for fast html not squad fitness

In [ ]:
import matplotlib.pyplot as plt

could we use matplotlib to show distributions for squad.__ft__ maybe by piece type?

lets do box plots. maybe each chart is one of health, food, distance and the boxes are the pieces? we could have separate lines for counts of how many are alive

I like both food and turns remaining so lets do 4 plots.
and yes lets do def __ft__(self, grid=None)) (and if no grid, no distance) we might also want to have a listPieces as a flag (which would give us our pieces list). sometimes we might just want the compact version without pieces so lets make that the default.

show(squad) to just work the no-args version gives you the compact 3-panel view is going to be great for our debugging.

can you write the full __ft__ for squad?

perfect. now lets go back to our simulator. can you build it?

In [ ]:
#| export

class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    RECEIVED  = "received"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"

@dataclass
class SimEvent:
    turn: int
    piece_id: str
    piece_name: str
    event: SimEventType
    hex_idx: int = -1
    detail: str = ""
    amount: float = 0.0

@dataclass
class FoodSimulator:
    """Turn-based simulator: move → gather → share → eat."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list                           # list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    events: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    # ── helpers ──────────────────────────────────────────────

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    def _log(self, piece, etype, detail="", amount=0.0):
        self.events.append(SimEvent(
            self.turn, piece.id, piece.name, etype,
            piece.location if piece.location is not None else -1,
            detail, amount))

    def _sort_move(self, pieces):
        """Rank → oldest → healthiest → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,        # ascending = oldest first
            -p.health,           # descending = healthiest first
            random.random()))

    def _sort_share(self, pieces):
        """Rank → oldest → least food → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            p.food,              # ascending = hungriest first
            random.random()))

    # ── move phase ───────────────────────────────────────────

    def _move_one(self, piece, occupied):
        """One turn of movement for a single piece. Returns list of executed Instructions."""
        il = piece.instructions
        if not il.rules:
            return []

        executed = []
        budget = float(piece.move_strength)

        while budget > 0:
            if il.cursor >= len(il.rules):
                if il.patrol:
                    il.cursor = 0
                else:
                    break

            instr = Instruction(il.rules[il.cursor])

            # ── free rotations ──
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                executed.append(instr)
                self._log(piece, SimEventType.ROTATED, instr.name)
                il.cursor += 1
                continue

            # ── cost-1 non-movement ──
            if instr in _COSTS_ONE:
                budget -= 1.0
                executed.append(instr)
                il.cursor += 1
                continue

            # ── FORWARD ──
            auto_rots = 0
            saved_fac = piece.facing
            moved = False

            while auto_rots <= 5:
                d = HexPosition.directions()[piece.facing % 6]
                nbr = self.grid.hexposition_to_index(d, piece.location)

                passable = (0 <= nbr < len(self.elevations)
                            and nbr not in self.grid.invalidRegion
                            and self.elevations[nbr] >= 1)
                if self.countries is not None and passable:
                    passable = self.countries[nbr] >= 0

                if not passable:
                    piece.facing = (piece.facing - 1) % 6
                    auto_rots += 1
                    continue

                # Occupied → freeze cursor, end turn
                if nbr in occupied and nbr != piece.location:
                    self._log(piece, SimEventType.BLOCKED, f"hex {nbr} occupied")
                    moved = True
                    budget = 0
                    break

                # Can afford?
                cost = _move_cost(self.elevations, piece.location, nbr)
                if cost > budget:
                    budget = 0
                    moved = True
                    break

                # Move
                occupied.discard(piece.location)
                piece.location = nbr
                occupied.add(nbr)
                budget -= cost
                executed.append(instr)
                self._log(piece, SimEventType.MOVED, f"→ hex {nbr}", cost)
                il.cursor += 1
                moved = True
                break

            if not moved:
                piece.facing = saved_fac
                self._log(piece, SimEventType.BLOCKED, "surrounded")
                budget = 0

        return executed

    def _move_phase(self):
        occupied = self.occupied
        order = self._sort_move(self.alive)
        executed = {}
        for piece in order:
            executed[piece.id] = self._move_one(piece, occupied)
        return executed

    # ── gather phase ─────────────────────────────────────────

    def _gather_phase(self, executed):
        for piece in self.alive:
            if Instruction.HARVEST in executed.get(piece.id, []):
                loc = piece.location
                if loc is None or loc < 0:
                    continue
                tier = int(self.food_tiers[loc])
                gained = piece.harvest_strength * tier
                old = piece.food
                piece.food = min(piece.food + gained, piece.food_capacity)
                self._log(piece, SimEventType.HARVESTED, f"tier {tier}", piece.food - old)

    # ── share phase ──────────────────────────────────────────

    def _share_phase(self, executed):
        givers = [p for p in self.alive
                  if Instruction.GIVE in executed.get(p.id, [])]

        for giver in self._sort_share(givers):
            reserve = FOOD_RESERVE_TURNS * giver.diet
            surplus = max(0.0, giver.food - reserve)
            if surplus <= 0:
                continue

            visible = set(giver.hexes_in_sight(
                self.grid, self.elevations,
                elevation_mult=self.elevation_mult,
                facing_only=True))

            # Re-sort receivers fresh each time (food levels change)
            receivers = sorted(
                [p for p in self.alive
                 if p is not giver
                 and p.owner_id == giver.owner_id
                 and p.food < p.food_capacity
                 and p.location in visible],
                key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

            for recv in receivers:
                if surplus <= 0:
                    break
                amount = min(surplus, recv.food_capacity - recv.food)
                recv.food += amount
                giver.food -= amount
                surplus -= amount
                self._log(giver, SimEventType.GAVE, f"→ {recv.name}", amount)
                self._log(recv, SimEventType.RECEIVED, f"← {giver.name}", amount)

    # ── eat phase ────────────────────────────────────────────


    # ── snapshot ─────────────────────────────────────────────

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive),
            'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0,
        })

 

    def run(self, num_turns=20):
        """Run multiple turns. Stops early if everyone is dead."""
        for _ in range(num_turns):
            if not self.alive:
                self._snapshot()
                break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")

    def events_for_turn(self, t): return [e for e in self.events if e.turn == t]
    def events_for_piece(self, pid): return [e for e in self.events if e.piece_id == pid]


In [ ]:
#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts: alive count, health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    df = pd.DataFrame(self.snapshots)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    # Alive
    axes[0].plot(df['turn'], df['alive'], 'o-', color='#2ecc71', lw=2, ms=4)
    axes[0].axhline(df['total'].iloc[0], color='#bbb', ls='--', lw=1)
    axes[0].set_title('Alive', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)
    axes[0].fill_between(df['turn'], df['alive'], alpha=0.15, color='#2ecc71')

    # Health
    axes[1].plot(df['turn'], df['avg_health'], 'o-', color='#e74c3c', lw=2, ms=4, label='avg')
    axes[1].plot(df['turn'], df['min_health'], 's--', color='#c0392b', lw=1, ms=3, alpha=0.6, label='min')
    axes[1].set_title('Health', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

    # Food turns remaining
    axes[2].plot(df['turn'], df['avg_turns_left'], 'o-', color='#f39c12', lw=2, ms=4)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left)', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].legend(fontsize=8); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")


I am also thinking about an __ft__ on FoodYield that would do matplotlib to show distributions

In [ ]:
#| export
@patch
def _eat_phase(self:FoodSimulator, executed):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue
        # Inactive this turn → excluded entirely (no eating)
        if piece.id not in executed or not executed[piece.id]:
            continue
            
        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, f"diet={piece.diet:.1f}", piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, f"dmg={damage:.1f}", damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
#| export
@patch
def step(self:FoodSimulator):
    executed = self._move_phase()
    self._gather_phase(executed)
    self._share_phase(executed)
    self._eat_phase(executed)      # ← now gated
    self._snapshot()
    self.turn += 1

sim = FoodSimulator(grid, elevs, food_tiers, all_pieces)
sim.run(30)
show(sim)        # instant line charts
sim.summary()    # text stats


## Demo and balancing

I am going to have two of the pieces be our primary food network suppliers. 

1. pawns are going to harvest but have low everything else.
2. bishops are going to be our transports. high capacity and moves. low everything else. I envision bishops going up and down supply lines.

There is also a question of different number of moves. so a piece might not have as many moves for a given turn - ie bishops will have more moves than pawns. for the steps that a pawn is missing it doesn't eat. I wanted to make sure that was in the simulator.

here is also a question of different number of moves. so a piece might not have as many moves for a given turn - ie bishops will have more moves than pawns. for the steps that a pawn is missing it doesn't eat. I wanted to make sure that was in the simulator.

Can we configure the pawn, bishop and queen food properties. The queen is hungry

## Food Profile

In [ ]:
#| export
@dataclass
class FoodProfile:
    """Per-piece-type food configuration."""
    diet:             float
    food_capacity:    float
    harvest_strength: int

    @staticmethod
    def apply(piece: 'Piece'):
        profile = FOOD_PROFILES.get(piece.piece_type)
        if profile and piece.diet == 1.0 and piece.food_capacity == 10.0:
            piece.diet             = profile.diet
            piece.food_capacity    = profile.food_capacity
            piece.harvest_strength = profile.harvest_strength
            piece.food             = profile.diet * FOOD_RESERVE_TURNS


FOOD_PROFILES: dict[PieceType, FoodProfile] = {
    PieceType.KING:   FoodProfile(diet=4.0, food_capacity=15.0, harvest_strength=1),
    PieceType.QUEEN:  FoodProfile(diet=3.0, food_capacity=12.0, harvest_strength=2),
    PieceType.ROOK:   FoodProfile(diet=2.0, food_capacity=12.0, harvest_strength=1),
    PieceType.BISHOP: FoodProfile(diet=1.5, food_capacity=25.0, harvest_strength=1),
    PieceType.KNIGHT: FoodProfile(diet=1.5, food_capacity=10.0, harvest_strength=1),
    PieceType.PAWN:   FoodProfile(diet=0.5, food_capacity=8.0,  harvest_strength=3),
}


can you build a demo of a hungry queen away from a group of pawns but is being fed by bishops? Maybe have it run for a few turns?

I need to fix some box plots: I am getting /tmp/ipykernel_1840/3142545611.py:74: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=type_labels, patch_artist=True,

In [ ]:
_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.HARVEST, Instruction.GIVE, Instruction.PAUSE}


In [ ]:
# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Simple food tiers: low ground = fertile, high = barren
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0   # ocean
    elif e < 60:    food_tiers[i] = 7   # coastal plains  ← pawns go here
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0   # peaks

def find_land(origin, offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, origin)
        if idx >= 0 and idx not in grid.invalidRegion and elevs[idx] > 0:
            return idx
    return None

# ── Queen — high ground, far from food, starts hungry ──────────────────────
queen_idx = find_land(mid, [HexPosition(q, r, -q-r)
                             for q in range(-2, 3) for r in range(-2, 3)
                             if elevs[grid.hexposition_to_index(
                                 HexPosition(q,r,-q-r), mid)] > 200])

if queen_idx is None:   # fallback
    queen_idx = find_land(mid, [HexPosition(0,0,0)])

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0,
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4.5   # only 1.5 turns of food — very hungry

print(f"Queen @ hex {queen_idx}  elev={elevs[queen_idx]:.0f}  "
      f"food={queen.food:.1f}/{queen.food_capacity}  diet={queen.diet}")

# ── Pawns — fertile low tiles, harvest+give loop ────────────────────────────
pawn_offsets = [
    HexPosition(-5, 3, 2), HexPosition(-4, 4, 0), HexPosition(-6, 4, 2),
    HexPosition(-5, 5,-0), HexPosition(-3, 3, 0),
]
pawns = []
for hp in pawn_offsets:
    idx = find_land(mid, [hp, hp + HexPosition(1,0,-1), hp + HexPosition(0,1,-1)])
    if idx is None or food_tiers[idx] < 4:
        continue
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=random.randint(0,5), country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value,
                             Instruction.GIVE.value], patrol=True))
    FoodProfile.apply(p)
    p.food = p.diet * FOOD_RESERVE_TURNS
    pawns.append(p)

print(f"{len(pawns)} pawns on food tiers: "
      f"{[food_tiers[p.location] for p in pawns]}")

# ── Bishops — run routes between pawn cluster and queen ─────────────────────
# Place two bishops midway between the pawn cluster centroid and the queen
pawn_positions = [grid.index_to_hexposition(p.location) for p in pawns]
pc_q = int(np.mean([p.q for p in pawn_positions]))
pc_r = int(np.mean([p.r for p in pawn_positions]))

queen_pos = grid.index_to_hexposition(queen_idx)

# Two waypoints: 1/3 and 2/3 along the pawn→queen vector
bishops = []
for frac, label in [(0.35, "near"), (0.65, "far")]:
    bq = int(pc_q + frac * (queen_pos.q - pc_q))
    br = int(pc_r + frac * (queen_pos.r - pc_r))
    bs = -bq - br
    b_idx = find_land(mid, [HexPosition(bq, br, bs),
                             HexPosition(bq+1, br, bs-1),
                             HexPosition(bq, br+1, bs-1),
                             HexPosition(bq-1, br, bs+1)])
    if b_idx is None:
        continue

    # Bishop path: pawn cluster → queen → back (supply run)
    pawn_anchor = pawns[0].location
    path_out = board.pieces[0].pathfind.__func__(  # use a dummy pathfind
        type('_', (), {'location': b_idx, 'pathfind': lambda s,t,g,e,c=None:
            None})(), queen_idx, grid, elevs) \
        if False else None

    # Simpler: give bishop a HARVEST+GIVE patrol so it passes food along
    b = board.add_piece(b_idx, PieceType.BISHOP,
                        facing=_facing_toward(grid, b_idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.GIVE.value,
                             Instruction.FORWARD.value,
                             Instruction.GIVE.value,
                             Instruction.FORWARD.value], patrol=True))
    FoodProfile.apply(b)
    b.food = b.diet * FOOD_RESERVE_TURNS
    bishops.append(b)
    print(f"  Bishop ({label}) @ hex {b_idx}  elev={elevs[b_idx]:.0f}  "
          f"capacity={b.food_capacity}")

# ── Pathfind bishops toward queen properly ──────────────────────────────────
for b in bishops:
    path = b.pathfind(queen_idx, grid, elevs)
    if path and len(path) > 1:
        rules = (path.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + list(reversed(path.to_rules(grid, b.facing)))
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"  Bishop route: {len(path)} hexes, "
              f"{len(b.instructions.rules)} instructions")

# ── Squads for display ───────────────────────────────────────────────────────
royal_guard = Squad(id=1, name="Royal Guard",
                    pieces=[queen] + bishops)
supply_corps = Squad(id=2, name="Supply Corps",
                     pieces=pawns)

all_pieces = [queen] + bishops + pawns

# ── Run simulator ────────────────────────────────────────────────────────────
# ── Before state ─────────────────────────────────────────────────────────────
print("=== BEFORE ===")
sim = FoodSimulator(
    grid=grid,
    elevations=elevs,
    food_tiers=food_tiers,
    pieces=all_pieces,
)
sim.summary()

#show(royal_guard)
#show(supply_corps)
#show(NotStr(board.render_with_facing(num_turns=12)))

sim.run(num_turns=25)

print("\nAfter:")
#sim.summary()
print(f"\nQueen food: {queen.food:.1f}  health: {queen.health:.0f}")
print(f"Events for queen: {len(sim.events_for_piece(queen.id))}")

if False:
    
    show(royal_guard)
    show(supply_corps)
    show(NotStr(board.render_with_sight(
        [queen] + bishops + pawns,
        num_turns=12,
        facing_only=True
    )))



# Sample queen's event log
for ev in sim.events_for_piece(queen.id)[:12]:
    print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  "
          f"{'±'+str(round(ev.amount,1)) if ev.amount else ''}")


We lose a pawn?

can you show the inital board with the pieces plans so I can see how it is going?

So the queen is on the opposite side of the river outside of where the bishops can see. So she can't be fed

In [ ]:
# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0
    elif e < 60:    food_tiers[i] = 7
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# ── Step 1: Find a fertile cluster for pawns ───────────────────────────────
# Pick the best food hex near the center as our anchor
best_food = sorted(
    [i for i in range(len(elevs))
     if elevs[i] > 0 and i not in grid.invalidRegion and food_tiers[i] >= 5],
    key=lambda i: abs(grid.index_to_hexposition(i, mid))
)[:20]  # 20 closest fertile hexes

pawn_anchor = best_food[0]
print(f"Pawn anchor: hex {pawn_anchor}  tier={food_tiers[pawn_anchor]}  elev={elevs[pawn_anchor]:.0f}")

# ── Step 2: Find queen spot REACHABLE from pawn anchor, on higher ground ────
# Dijkstra outward from pawn_anchor, pick first hex with elev > 150
dummy = Piece(location=pawn_anchor)
queen_idx = None
for min_elev in [180, 150, 120, 80]:
    # Search outward for elevated reachable hex
    pq = [(0.0, pawn_anchor)]
    visited = set()
    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited.add(cur)
        if cur != pawn_anchor and elevs[cur] >= min_elev and food_tiers[cur] <= 2:
            queen_idx = cur
            break
        for nb in grid.neighborsOf(cur):
            if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
            heapq.heappush(pq, (cost + _move_cost(elevs, cur, nb), nb))
    if queen_idx: break

print(f"Queen: hex {queen_idx}  elev={elevs[queen_idx]:.0f}  tier={food_tiers[queen_idx]}")

# Verify path exists
test_path = dummy.pathfind(queen_idx, grid, elevs)
print(f"Path pawn→queen: {len(test_path)} hexes, cost={test_path.cost:.1f}")
assert test_path, "No path — pick different spots!"

# ── Step 3: Place pieces ──────────────────────────────────────────────────
queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4.5

# Pawns on best food hexes near anchor
pawns = []
for idx in best_food[:5]:
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=_facing_toward(grid, idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value, Instruction.GIVE.value],
                            patrol=True))
    FoodProfile.apply(p)
    pawns.append(p)

# Bishops — start near pawns, loaded with food, route to queen and back
bishops = []
for i, start_near in enumerate(best_food[5:7]):
    b = board.add_piece(start_near, PieceType.BISHOP,
                        facing=_facing_toward(grid, start_near, queen_idx),
                        country_id=1,
                        instructions=InstructionList([]))
    FoodProfile.apply(b)
    b.food = b.food_capacity  # fully loaded courier

    # Route: bishop → queen → back to start
    path_out = b.pathfind(queen_idx, grid, elevs)
    path_back = Piece(location=queen_idx).pathfind(start_near, grid, elevs)

    if path_out and path_back:
        rules = (path_out.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + path_back.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"Bishop {i}: hex {start_near} → queen ({len(path_out)} hops) "
              f"→ back ({len(path_back)} hops), {len(rules)} instructions")
    bishops.append(b)

all_pieces = [queen] + bishops + pawns

# ── Show the plan ──────────────────────────────────────────────────────────
if False:
    show(NotStr(board.render_with_sight(
        all_pieces, num_turns=12, facing_only=True)))


# ── Run simulator ────────────────────────────────────────────────────────────
# ── Before state ─────────────────────────────────────────────────────────────
print("=== BEFORE ===")
sim = FoodSimulator(
    grid=grid,
    elevations=elevs,
    food_tiers=food_tiers,
    pieces=all_pieces,
)
sim.summary()

#show(royal_guard)
#show(supply_corps)
#show(NotStr(board.render_with_facing(num_turns=12)))

sim.run(num_turns=25)

print("\nAfter:")
#sim.summary()
print(f"\nQueen food: {queen.food:.1f}  health: {queen.health:.0f}")
print(f"Events for queen: {len(sim.events_for_piece(queen.id))}")

if False:
    
    show(royal_guard)
    show(supply_corps)
    show(NotStr(board.render_with_sight(
        [queen] + bishops + pawns,
        num_turns=12,
        facing_only=True
    )))



# Sample queen's event log
for ev in sim.events_for_piece(queen.id)[:12]:
    print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  "
          f"{'±'+str(round(ev.amount,1)) if ev.amount else ''}")

The queen lives?

lets push the queen further away. She needs to drop a few pounds

# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0
    elif e < 60:    food_tiers[i] = 7
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# ── Step 1: Fertile cluster for pawns ──────────────────────────────────────
best_food = sorted(
    [i for i in range(len(elevs))
     if elevs[i] > 0 and i not in grid.invalidRegion and food_tiers[i] >= 5],
    key=lambda i: abs(grid.index_to_hexposition(i, mid))
)[:20]

pawn_anchor = best_food[0]
print(f"Pawn anchor: hex {pawn_anchor}  tier={food_tiers[pawn_anchor]}  elev={elevs[pawn_anchor]:.0f}")

# ── Step 2: Queen far away — minimum 6 hops from pawn anchor ───────────────
MIN_HOPS = 6  # ← the diet plan

dummy = Piece(location=pawn_anchor)
queen_idx = None
candidates = []

pq = [(0.0, pawn_anchor)]
visited = {}
parent = {pawn_anchor: None}

while pq:
    cost, cur = heapq.heappop(pq)
    if cur in visited: continue
    visited[cur] = cost
    
    # Count hops via parent chain
    hops = 0
    node = cur
    while parent.get(node) is not None:
        hops += 1
        node = parent[node]
    
    # Candidate: far enough, elevated, low food (she's on a hill, not a farm)
    if hops >= MIN_HOPS and elevs[cur] >= 120 and food_tiers[cur] <= 2:
        candidates.append((cur, hops, cost, elevs[cur]))
    
    for nb in grid.neighborsOf(cur):
        if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
        new_cost = cost + _move_cost(elevs, cur, nb)
        if nb not in parent or new_cost < visited.get(nb, float('inf')):
            parent[nb] = cur
            heapq.heappush(pq, (new_cost, nb))

# Pick the highest elevation among far candidates
if candidates:
    candidates.sort(key=lambda c: (-c[1], -c[3]))  # most hops, then highest
    queen_idx, q_hops, q_cost, q_elev = candidates[0]

print(f"Queen: hex {queen_idx}  elev={q_elev:.0f}  tier={food_tiers[queen_idx]}  "
      f"hops={q_hops}  path_cost={q_cost:.1f}")

test_path = dummy.pathfind(queen_idx, grid, elevs)
print(f"Verified path: {len(test_path)} hexes, cost={test_path.cost:.1f}")

# ── Step 3: Place pieces ──────────────────────────────────────────────────
queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4  # modest reserves

# Pawns
pawns = []
for idx in best_food[:5]:
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=_facing_toward(grid, idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value, Instruction.GIVE.value],
                            patrol=True))
    FoodProfile.apply(p)
    pawns.append(p)

# Bishops — 3 this time for the longer route
bishops = []
for i, start_near in enumerate(best_food[5:8]):
    b = board.add_piece(start_near, PieceType.BISHOP,
                        facing=_facing_toward(grid, start_near, queen_idx),
                        country_id=1,
                        instructions=InstructionList([]))
    FoodProfile.apply(b)
    b.food = b.food_capacity  # fully loaded

    path_out = b.pathfind(queen_idx, grid, elevs)
    path_back = Piece(location=queen_idx).pathfind(start_near, grid, elevs)

    if path_out and path_back:
        rules = (path_out.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + path_back.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"  Bishop {i}: {len(path_out)} hops out, "
              f"{len(path_back)} back, {len(rules)} instructions")
    bishops.append(b)

all_pieces = [queen] + bishops + pawns

# ── Render plan ──────────────────────────────────────────────────────────
show(NotStr(board.render_with_sight(
    all_pieces, num_turns=15, facing_only=True)))

# ── Simulate ─────────────────────────────────────────────────────────────
sim = FoodSimulator(grid=grid, elevations=elevs,
                    food_tiers=food_tiers, pieces=all_pieces)

print(f"\n=== BEFORE === queen food={queen.food:.1f} diet={queen.diet}")
sim.run(num_turns=30)
sim.summary()

print(f"\nQueen: food={queen.food:.1f}  health={queen.health:.0f}")
show(sim)

# Queen's story
print("\nQueen's log:")
for ev in sim.events_for_piece(queen.id):
    if ev.event in (SimEventType.RECEIVED, SimEventType.STARVED, SimEventType.DIED):
        print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  ±{ev.amount:.1f}")


She dies

This leads into harvest planning. we need algorithm to figure out how far we can go from our settlement if we have x pieces to build a supply line

In [ ]:
#| export
@dataclass
class SupplyPlanner:
    terrain: Terrain
    settlement: int
    profiles: dict = field(default_factory=lambda: FOOD_PROFILES)

    @property
    def grid(self): return self.terrain.hexGrid
    @property
    def elevations(self): return self.terrain.elevations
    @property
    def food_tiers(self):
        return self.terrain.fields.get('food_yield',
            np.zeros(len(self.terrain.elevations), dtype=int))


    @property
    def bishop(self): return self.profiles[PieceType.BISHOP]
    @property
    def pawn(self): return self.profiles[PieceType.PAWN]
    @property
    def queen(self): return self.profiles[PieceType.QUEEN]

    # ── Core math ────────────────────────────────────────────

    def travel_turns(self, path_cost, move_strength=None):
        """How many turns to traverse a path of given cost."""
        ms = move_strength or 3  # bishop default
        return math.ceil(path_cost / ms)

    def net_delivery(self, path_cost, move_strength=None):
        """Food a bishop delivers after eating its own share on the round trip."""
        tt = self.travel_turns(path_cost, move_strength)
        eaten = 2 * tt * self.bishop.diet
        return max(0.0, self.bishop.food_capacity - eaten)

    def throughput(self, path_cost, n_bishops=1, move_strength=None):
        """Food delivered per turn with N bishops on this route."""
        tt = self.travel_turns(path_cost, move_strength)
        nd = self.net_delivery(path_cost, move_strength)
        if nd <= 0 or tt <= 0:
            return 0.0
        round_trip = 2 * tt
        return n_bishops * nd / round_trip

    def bishops_needed(self, path_cost, consumer_diet=None, move_strength=None):
        """Minimum bishops to sustain a consumer (default: queen) at this distance."""
        diet = consumer_diet or self.queen.diet
        tp1 = self.throughput(path_cost, n_bishops=1, move_strength=move_strength)
        if tp1 <= 0:
            return float('inf')  # unreachable
        return math.ceil(diet / tp1)

    def max_range(self, move_strength=None):
        """Maximum one-way path cost where a bishop can still deliver food."""
        ms = move_strength or 3
        max_tt = self.bishop.food_capacity / (2 * self.bishop.diet)
        return int(max_tt * ms)

    def startup_food(self, path_cost, move_strength=None):
        """How much food the consumer needs to survive until first delivery."""
        tt = self.travel_turns(path_cost, move_strength)
        return self.queen.diet * (tt + 1)  # +1 buffer turn

    # ── Harvest rate at settlement ───────────────────────────

    def harvest_rate(self, n_pawns=5):
        """Food per turn from N pawns at the settlement."""
        # Average tier of nearby fertile hexes
        nbs = [self.settlement] + list(self.grid.neighborsOf(self.settlement))
        tiers = [self.food_tiers[n] for n in nbs
                 if 0 <= n < len(self.food_tiers) and self.food_tiers[n] > 0]
        avg_tier = np.mean(tiers) if tiers else 0
        return n_pawns * self.pawn.harvest_strength * avg_tier

    def can_sustain(self, n_pawns, n_bishops, path_cost, n_consumers=1,
                    consumer_diet=None, move_strength=None):
        """Can this setup sustain N consumers at the given distance?"""
        diet = consumer_diet or self.queen.diet
        total_demand = n_consumers * diet + n_bishops * self.bishop.diet
        supply = self.harvest_rate(n_pawns)
        delivery = self.throughput(path_cost, n_bishops, move_strength)
        return supply >= total_demand and delivery >= n_consumers * diet

    # ── Reachability map ─────────────────────────────────────

    def supply_range_map(self, move_strength=None):
        """Dijkstra from settlement → dict of {hex: (path_cost, bishops_needed)}."""
        max_cost = self.max_range(move_strength)
        pq = [(0.0, self.settlement)]
        visited = {}

        while pq:
            cost, cur = heapq.heappop(pq)
            if cur in visited:
                continue
            visited[cur] = (cost, self.bishops_needed(cost, move_strength=move_strength))

            for nb in self.grid.neighborsOf(cur):
                if nb in self.grid.invalidRegion or self.elevations[nb] <= 0:
                    continue
                new_cost = cost + _move_cost(self.elevations, cur, nb)
                if new_cost <= max_cost and nb not in visited:
                    heapq.heappush(pq, (new_cost, nb))

        return visited

    def summary(self, n_pawns=5, n_bishops=2):
        """Print a quick supply line summary."""
        mr = self.max_range()
        hr = self.harvest_rate(n_pawns)
        print(f"Settlement hex {self.settlement}  "
              f"tier={self.food_tiers[self.settlement]}")
        print(f"  Max bishop range: {mr} path cost "
              f"({self.travel_turns(mr)} turns one-way)")
        print(f"  Harvest rate: {hr:.1f} food/turn ({n_pawns} pawns)")
        print(f"  Bishop capacity: {self.bishop.food_capacity}, "
              f"diet: {self.bishop.diet}/turn")
        print()
        for cost in range(2, mr + 1, 2):
            nd = self.net_delivery(cost)
            tp = self.throughput(cost, n_bishops)
            bn = self.bishops_needed(cost)
            sf = self.startup_food(cost)
            print(f"  cost={cost:2d}  net={nd:5.1f}  "
                  f"throughput={tp:.2f}/turn  "
                  f"bishops_needed={bn}  startup_food={sf:.0f}")


In [ ]:
#| export
@patch
def plan_route(self: SupplyPlanner, target, board=None):
    """Analyze a supply route to a target hex. Returns dict of metrics."""
    dummy = Piece(location=self.settlement)
    path = dummy.pathfind(target, self.grid, self.elevations)
    if not path:
        return None
    cost = path.cost
    return {
        'target': target,
        'path': path,
        'cost': cost,
        'travel_turns': self.travel_turns(cost),
        'net_delivery': self.net_delivery(cost),
        'bishops_needed': self.bishops_needed(cost),
        'startup_food': self.startup_food(cost),
        'throughput_per_bishop': self.throughput(cost, 1),
        'reachable': self.net_delivery(cost) > 0,
    }

@patch
def find_target(self: SupplyPlanner, min_elev=120, max_tier=2,
                min_cost=4.0, max_cost=None):
    """Find a reachable target hex: elevated, low food, within supply range."""
    max_cost = max_cost or self.max_range()
    pq = [(0.0, self.settlement)]
    visited = {}
    candidates = []

    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited[cur] = cost

        if (cur != self.settlement
            and self.elevations[cur] >= min_elev
            and self.food_tiers[cur] <= max_tier
            and cost >= min_cost):
            candidates.append((cur, cost, self.elevations[cur]))

        for nb in self.grid.neighborsOf(cur):
            if nb in self.grid.invalidRegion or self.elevations[nb] <= 0: continue
            new_cost = cost + _move_cost(self.elevations, cur, nb)
            if new_cost <= max_cost and nb not in visited:
                heapq.heappush(pq, (new_cost, nb))

    if not candidates:
        return None
    # Highest elevation among reachable candidates
    candidates.sort(key=lambda c: (-c[2], c[1]))
    idx, cost, elev = candidates[0]
    return idx

@patch
def create_supply_line(self: SupplyPlanner, target, board,
                       n_bishops=None, country_id=1):
    """Place and configure bishops on a supply route. Returns list of Pieces."""
    route = self.plan_route(target)
    if not route or not route['reachable']:
        print(f"⚠ Target hex {target} is unreachable from settlement {self.settlement}")
        return []

    n_bishops = n_bishops or route['bishops_needed']
    path_out = route['path']

    # Find start hexes near settlement for each bishop
    # Spread them across fertile hexes so they don't stack
    nearby = sorted(
        [i for i in [self.settlement] + list(self.grid.neighborsOf(self.settlement))
         if i not in self.grid.invalidRegion and self.elevations[i] > 0],
        key=lambda i: -self.food_tiers[i]  # prefer fertile start hexes
    )

    bishops = []
    occupied = {p.location for p in board.pieces if p.location is not None}

    for i in range(n_bishops):
        # Pick a start hex that isn't already occupied
        start = None
        for candidate in nearby:
            if candidate not in occupied:
                start = candidate
                break
        if start is None:
            start = nearby[i % len(nearby)]  # fallback: stack if needed

        b = board.add_piece(
            start, PieceType.BISHOP,
            facing=_facing_toward(self.grid, start, target),
            country_id=country_id,
            instructions=InstructionList([]))
        FoodProfile.apply(b)
        b.food = b.food_capacity  # fully loaded courier

        # Build route: outbound → GIVE → return → GIVE (pick up from pawns)
        path_back = Piece(location=target).pathfind(start, self.grid, self.elevations)
        if path_out and path_back:
            rules = (path_out.to_rules(self.grid, b.facing)
                     + [Instruction.GIVE.value]
                     + path_back.to_rules(self.grid, b.facing)
                     + [Instruction.GIVE.value])
            b.instructions = InstructionList(rules, cursor=0, patrol=True)

        occupied.add(start)
        bishops.append(b)

    print(f"🚚 Supply line: {n_bishops} bishops, "
          f"cost={route['cost']:.1f}, "
          f"{route['travel_turns']} turns one-way, "
          f"net delivery={route['net_delivery']:.1f}/trip")
    return bishops

@patch
def provision_consumer(self: SupplyPlanner, piece, target=None, path_cost=None):
    """Set a consumer's starting food to survive until first delivery."""
    if path_cost is None and target is not None:
        route = self.plan_route(target)
        path_cost = route['cost'] if route else 0
    food = self.startup_food(path_cost or 0)
    piece.food = min(food, piece.food_capacity)
    print(f"🍖 {piece.name} provisioned with {piece.food:.1f} food "
          f"(need {food:.0f} to survive delivery wait)")


In [ ]:
#| export
@patch
def __ft__(self: SupplyPlanner):
    """Supply range heatmap from settlement + throughput curve."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    # ── 1. Throughput vs distance curve ──────────────────────
    ax = axes[0]
    mr = self.max_range()
    costs = np.arange(1, mr + 1, 0.5)
    tp1 = [self.throughput(c, 1) for c in costs]
    tp2 = [self.throughput(c, 2) for c in costs]
    tp3 = [self.throughput(c, 3) for c in costs]

    ax.plot(costs, tp1, '-',  color='#3498db', lw=2, label='1 bishop')
    ax.plot(costs, tp2, '--', color='#2ecc71', lw=2, label='2 bishops')
    ax.plot(costs, tp3, ':',  color='#e67e22', lw=2, label='3 bishops')
    ax.axhline(self.queen.diet, color='#e74c3c', ls='-.', lw=1.5,
               alpha=0.7, label=f'queen diet ({self.queen.diet})')
    ax.axhline(self.profiles[PieceType.KNIGHT].diet, color='#9b59b6',
               ls='-.', lw=1, alpha=0.5, label=f'knight diet')
    ax.set_title('Throughput vs Distance', fontweight='bold', fontsize=10)
    ax.set_xlabel('Path cost')
    ax.set_ylabel('Food/turn delivered')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='both', alpha=0.3, lw=0.5)

    # ── 2. Bishops needed heatmap by (cost, consumer_diet) ───
    ax = axes[1]
    diets = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
    cost_range = list(range(2, mr + 1, 2))
    grid_data = []
    for d in diets:
        row = []
        for c in cost_range:
            bn = self.bishops_needed(c, consumer_diet=d)
            row.append(min(bn, 10))  # cap for display
        grid_data.append(row)

    im = ax.imshow(grid_data, aspect='auto', cmap='YlOrRd',
                   vmin=0, vmax=8, origin='lower')
    ax.set_xticks(range(len(cost_range)))
    ax.set_xticklabels(cost_range, fontsize=7)
    ax.set_yticks(range(len(diets)))
    ax.set_yticklabels([f'{d:.1f}' for d in diets], fontsize=7)
    ax.set_xlabel('Path cost')
    ax.set_ylabel('Consumer diet')
    ax.set_title('Bishops Needed', fontweight='bold', fontsize=10)

    # Annotate cells
    for i, d in enumerate(diets):
        for j, c in enumerate(cost_range):
            val = grid_data[i][j]
            txt = '∞' if val >= 10 else str(val)
            color = 'white' if val > 4 else 'black'
            ax.text(j, i, txt, ha='center', va='center',
                    fontsize=7, color=color, fontweight='bold')

    plt.colorbar(im, ax=ax, shrink=0.8, label='Bishops')

    hr = self.harvest_rate()
    fig.suptitle(
        f"Supply Planner — settlement hex {self.settlement}, "
        f"max range {mr}, harvest {hr:.0f}/turn",
        fontsize=10, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Bishop: cap={self.bishop.food_capacity} diet={self.bishop.diet} · "
          f"Pawn: harvest={self.pawn.harvest_strength}",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


I switched SupplyPlanner to use terain. does any of this need to be refactored?

better?

Should we have some linear program that optimizes which supply lines to create and who get them. we could have a max the colony kind of thing.

### ColonyOptimizer

In [ ]:
#| export
from scipy.optimize import milp, LinearConstraint, Bounds

@dataclass
class ColonyOptimizer:
    """Optimize supply line allocation across a colony."""
    planner: SupplyPlanner
    max_pawns: int = 8
    max_bishops: int = 6

    @dataclass
    class Consumer:
        piece: object           # Piece or just a name
        hex_idx: int
        diet: float
        priority: float = 1.0   # queen=10, knight=3, rook=2, etc.
        path_cost: float = 0.0
        reachable: bool = True

    def analyze_consumers(self, pieces):
        """Build Consumer list with route analysis."""
        consumers = []
        for p in pieces:
            route = self.planner.plan_route(p.location)
            if route is None:
                consumers.append(self.Consumer(
                    piece=p, hex_idx=p.location, diet=p.diet,
                    priority=_priority(p), path_cost=float('inf'),
                    reachable=False))
            else:
                consumers.append(self.Consumer(
                    piece=p, hex_idx=p.location, diet=p.diet,
                    priority=_priority(p), path_cost=route['cost'],
                    reachable=route['reachable']))
        return consumers

    def optimize(self, consumers):
        """
        Decide which consumers to feed and how many bishops per route.
        
        Returns dict with:
          - assignments: list of (consumer, n_bishops, throughput)
          - total_fed: number of consumers fed
          - total_priority: sum of priority of fed consumers
          - pawns_used, bishops_used
        """
        # Filter to reachable consumers
        reachable = [c for c in consumers if c.reachable and c.path_cost > 0]
        if not reachable:
            return self._empty_result(consumers)

        n = len(reachable)
        sp = self.planner

        # ── Variables ──────────────────────────────────────────
        # For each consumer i:
        #   b_i = bishops assigned (integer ≥ 0)
        #   f_i = binary: is this consumer fed?
        # Layout: [b_0, b_1, ..., b_n-1, f_0, f_1, ..., f_n-1]

        # Precompute per-route throughput per bishop
        tp1 = [sp.throughput(c.path_cost, 1) for c in reachable]

        # ── Objective: maximize Σ priority_i * f_i ─────────────
        # milp minimizes, so negate priorities
        c_obj = np.zeros(2 * n)
        for i, con in enumerate(reachable):
            c_obj[n + i] = -con.priority  # maximize priority

        # ── Variable bounds ────────────────────────────────────
        # b_i ∈ [0, max_bishops], f_i ∈ [0, 1]
        lb = np.zeros(2 * n)
        ub = np.concatenate([
            np.full(n, self.max_bishops),   # b_i upper
            np.ones(n)                       # f_i upper
        ])

        # ── Integrality: all integer ───────────────────────────
        integrality = np.ones(2 * n)  # 1 = integer

        # ── Constraints ────────────────────────────────────────
        A_rows = []
        lb_cons = []
        ub_cons = []

        # 1. Total bishops ≤ max_bishops
        row = np.zeros(2 * n)
        row[:n] = 1.0
        A_rows.append(row)
        lb_cons.append(0)
        ub_cons.append(self.max_bishops)

        # 2. For each consumer: tp1_i * b_i ≥ diet_i * f_i
        #    → tp1_i * b_i - diet_i * f_i ≥ 0
        for i, con in enumerate(reachable):
            row = np.zeros(2 * n)
            row[i] = tp1[i]           # bishop throughput
            row[n + i] = -con.diet    # diet requirement
            A_rows.append(row)
            lb_cons.append(0)
            ub_cons.append(np.inf)

        # 3. Total food demand ≤ harvest capacity
        #    Σ (diet_i * f_i) + Σ (bishop_diet * b_i) ≤ harvest_rate
        harvest = sp.harvest_rate(self.max_pawns)
        row = np.zeros(2 * n)
        for i, con in enumerate(reachable):
            row[i] = sp.bishop.diet       # bishop consumption
            row[n + i] = con.diet         # consumer consumption
        A_rows.append(row)
        lb_cons.append(0)
        ub_cons.append(harvest)

        # 4. f_i can only be 1 if at least 1 bishop assigned
        #    f_i ≤ b_i  →  -b_i + f_i ≤ 0
        for i in range(n):
            row = np.zeros(2 * n)
            row[i] = -1.0
            row[n + i] = 1.0
            A_rows.append(row)
            lb_cons.append(-np.inf)
            ub_cons.append(0)

        A = np.array(A_rows)
        constraints = LinearConstraint(A, lb_cons, ub_cons)
        bounds = Bounds(lb, ub)

        # ── Solve ──────────────────────────────────────────────
        result = milp(c_obj, constraints=constraints, bounds=bounds,
                      integrality=integrality)

        if not result.success:
            print(f"⚠ Optimization failed: {result.message}")
            return self._empty_result(consumers)

        x = np.round(result.x).astype(int)
        b_vals = x[:n]
        f_vals = x[n:]

        # ── Build result ───────────────────────────────────────
        assignments = []
        for i, con in enumerate(reachable):
            assignments.append({
                'consumer': con,
                'bishops': b_vals[i],
                'fed': bool(f_vals[i]),
                'throughput': tp1[i] * b_vals[i],
                'surplus': tp1[i] * b_vals[i] - con.diet if f_vals[i] else 0,
            })

        return {
            'assignments': assignments,
            'total_fed': int(f_vals.sum()),
            'total_priority': sum(a['consumer'].priority
                                  for a in assignments if a['fed']),
            'bishops_used': int(b_vals.sum()),
            'harvest_available': harvest,
            'harvest_consumed': sum(
                a['bishops'] * sp.bishop.diet + (a['consumer'].diet if a['fed'] else 0)
                for a in assignments),
        }

    def _empty_result(self, consumers):
        return {'assignments': [], 'total_fed': 0, 'total_priority': 0,
                'bishops_used': 0, 'harvest_available': 0, 'harvest_consumed': 0}

    def print_plan(self, result):
        """Pretty-print the optimization result."""
        print(f"═══ Colony Supply Plan ═══")
        print(f"  Fed: {result['total_fed']} consumers  "
              f"(priority score: {result['total_priority']:.0f})")
        print(f"  Bishops: {result['bishops_used']}/{self.max_bishops}")
        print(f"  Harvest: {result['harvest_consumed']:.1f}"
              f"/{result['harvest_available']:.1f} food/turn")
        print()
        for a in sorted(result['assignments'],
                        key=lambda x: -x['consumer'].priority):
            c = a['consumer']
            status = "✅" if a['fed'] else "❌"
            name = getattr(c.piece, 'name', str(c.hex_idx))
            print(f"  {status} {name:<16} "
                  f"pri={c.priority:<4.0f} "
                  f"diet={c.diet:<4.1f} "
                  f"cost={c.path_cost:<5.1f} "
                  f"bishops={a['bishops']} "
                  f"tp={a['throughput']:.2f}/turn")


def _priority(piece):
    """Default priority by piece type."""
    return {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.ROOK: 5, PieceType.KNIGHT: 3,
        PieceType.BISHOP: 2, PieceType.PAWN: 1,
    }.get(piece.piece_type, 1)


In [ ]:
#| export
@patch
def __ft__(self: ColonyOptimizer):
    """Visualize the last optimization result."""
    # We need a result to display — run a quick check
    if not hasattr(self, '_last_result') or not self._last_result:
        return P("No optimization run yet — call .optimize() first",
                 cls="text-sm opacity-50")

    result = self._last_result
    assignments = sorted(result['assignments'],
                         key=lambda a: -a['consumer'].priority)

    if not assignments:
        return P("No reachable consumers", cls="text-sm opacity-50")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    names = [getattr(a['consumer'].piece, 'name', f"hex {a['consumer'].hex_idx}")
             for a in assignments]
    n = len(names)
    x = np.arange(n)

    # ── 1. Bishops allocated per consumer ────────────────────
    ax = axes[0]
    bishops = [a['bishops'] for a in assignments]
    colors = ['#2ecc71' if a['fed'] else '#e74c3c' for a in assignments]
    ax.barh(x, bishops, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_yticks(x)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel('Bishops')
    ax.set_title('Bishop Allocation', fontweight='bold', fontsize=10)
    ax.axvline(0, color='#333', lw=0.5)
    ax.grid(axis='x', alpha=0.3)

    # ── 2. Throughput vs diet ────────────────────────────────
    ax = axes[1]
    throughputs = [a['throughput'] for a in assignments]
    diets = [a['consumer'].diet for a in assignments]

    bars = ax.barh(x, throughputs, color='#3498db', alpha=0.7,
                   edgecolor='#555', linewidth=0.5, label='throughput')
    # Overlay diet as markers
    for i, (tp, d) in enumerate(zip(throughputs, diets)):
        ax.plot(d, i, 'D', color='#e74c3c', ms=6, zorder=5)
    ax.plot([], [], 'D', color='#e74c3c', ms=6, label='diet')
    ax.set_yticks(x)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel('Food/turn')
    ax.set_title('Throughput vs Diet', fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(axis='x', alpha=0.3)

    # ── 3. Harvest budget pie ────────────────────────────────
    ax = axes[2]
    harvest = result['harvest_available']
    consumed = result['harvest_consumed']
    bishop_eat = sum(a['bishops'] * self.planner.bishop.diet
                     for a in assignments)
    consumer_eat = sum(a['consumer'].diet for a in assignments if a['fed'])
    spare = max(0, harvest - consumed)

    sizes = [consumer_eat, bishop_eat, spare]
    labels = [f'Consumers\n{consumer_eat:.1f}',
              f'Bishops\n{bishop_eat:.1f}',
              f'Spare\n{spare:.1f}']
    pie_colors = ['#e74c3c', '#3498db', '#95a5a6']

    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=pie_colors,
        autopct='%1.0f%%', startangle=90,
        textprops={'fontsize': 7})
    for t in autotexts:
        t.set_fontsize(7)
        t.set_fontweight('bold')
    ax.set_title('Harvest Budget', fontweight='bold', fontsize=10)

    fig.suptitle(
        f"Colony Plan — {result['total_fed']} fed, "
        f"{result['bishops_used']}/{self.max_bishops} bishops, "
        f"priority={result['total_priority']:.0f}",
        fontsize=10, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"✅ {result['total_fed']} fed · "
          f"❌ {len(assignments) - result['total_fed']} unfed · "
          f"🌾 {spare:.1f} food/turn spare",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


should ColonyOptimizer return some sort of battle plan that the user can accept parts of? My big quesion is how to make this interactive and entertaining?

This is quite a bit we have built up and I think this would be a good git checkpoint. That gets rid of the ai conversations so I need to put it into a note so you can think better about where we are headed and what we have built so far. I think these supply planner and colony optimizer are good foundations for a supply chain system. The visualizations are also amazing. This feels very different to any game I have played, but it does have a nice simple set of rules at is heart that I want to keep. Can you give me a good write up?

So my question for game design is what to we want the player to decide. I do think keeping the micromanagement is fine (if not cool and unqiue), but I am think we give them a heat map of where they might want to expand towards and they pick a few, and then we optimize. (For the AI they would do the steps themselves.)

Yes lets build a heat map. do we need multiple to show the trade offs?

In [ ]:
from HexMagic.primitives import HexGrid
??HexGrid

In [ ]:
??Terrain

I think we should start switching this to terrain. We quite literally have heat maps for it.

In [ ]:
#| export
@patch
def compute_supply_fields(self: SupplyPlanner):
    """Populate terrain.fields with supply metrics from settlement."""
    range_map = self.supply_range_map()  # {hex: (path_cost, bishops_needed)}
    n = len(self.elevations)

    costs      = np.full(n, np.nan)
    bishops    = np.full(n, np.nan)
    delivery   = np.full(n, 0.0)
    max_diet   = np.full(n, 0.0)
    efficiency = np.full(n, 0.0)

    for idx, (cost, bn) in range_map.items():
        costs[idx]      = cost
        bishops[idx]    = min(bn, 10)
        delivery[idx]   = self.net_delivery(cost)
        max_diet[idx]   = self.throughput(cost, 1)
        efficiency[idx] = delivery[idx] / cost if cost > 0 else 0

    self.terrain.fields['supply_cost']      = costs
    self.terrain.fields['bishops_needed']   = bishops
    self.terrain.fields['net_delivery']     = delivery
    self.terrain.fields['max_diet']         = max_diet
    self.terrain.fields['supply_efficiency'] = efficiency

    print(f"📡 Supply fields computed from hex {self.settlement}, "
          f"{len(range_map)} reachable hexes, max range {self.max_range()}")


In [ ]:
#| export
@patch
def render_supply_overlay(self: SupplyPlanner,
                          field='bishops_needed',
                          layer_name='supply_heatmap'):
    """Render a supply field onto the terrain using its SVG pipeline."""
    if field not in self.terrain.fields:
        self.compute_supply_fields()

    data = self.terrain.fields[field]
    grid = self.grid

    # Color stops per field type
    stops = {
        'bishops_needed':   [(1, "#2ecc71"), (2, "#f1c40f"), (4, "#e67e22"), (6, "#e74c3c"), (10, "#1a1a2e")],
        'net_delivery':     [(0, "#e74c3c"), (5, "#f1c40f"), (10, "#2ecc71"), (20, "#27ae60")],
        'max_diet':         [(0, "#e74c3c"), (1, "#f39c12"), (2, "#f1c40f"), (3, "#2ecc71"), (5, "#27ae60")],
        'supply_efficiency':[(0, "#e74c3c"), (0.5, "#f1c40f"), (1, "#2ecc71"), (2, "#27ae60")],
        'supply_cost':      [(0, "#2ecc71"), (5, "#f1c40f"), (10, "#e67e22"), (20, "#e74c3c")],
    }

    color_stops = stops.get(field, stops['supply_cost'])

    def lerp_color(val):
        """Interpolate between color stops."""
        if np.isnan(val): return None
        for i in range(len(color_stops) - 1):
            v0, c0 = color_stops[i]
            v1, c1 = color_stops[i + 1]
            if val <= v1:
                t = (val - v0) / (v1 - v0) if v1 != v0 else 0
                t = max(0, min(1, t))
                r0, g0, b0 = int(c0[1:3], 16), int(c0[3:5], 16), int(c0[5:7], 16)
                r1, g1, b1 = int(c1[1:3], 16), int(c1[3:5], 16), int(c1[5:7], 16)
                r = int(r0 + (r1 - r0) * t)
                g = int(g0 + (g1 - g0) * t)
                b = int(b0 + (b1 - b0) * t)
                return f"#{r:02x}{g:02x}{b:02x}"
        return color_stops[-1][1]

    # Build SVG overlay using the same hex polygon approach as render_icon_temperature
    overlay = ""
    for i in range(len(data)):
        if self.elevations[i] <= 0: continue  # skip ocean
        color = lerp_color(data[i])
        if color is None: continue

        hex_obj = grid.hexes[i]
        points = " ".join([f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices()])
        overlay += f'\t<polygon points="{points}" fill="{color}" opacity="0.5"/>\n'

    # Settlement marker
    s_hex = grid.hexes[self.settlement]
    sc = s_hex.center   # ← attribute, not method
    overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{grid.radius * 0.6}" '
                f'fill="none" stroke="#f1c40f" stroke-width="2.5" opacity="0.9"/>\n')


    grid.builder.adjust(layer_name, overlay)
    return self.terrain


Can you build a demo using myTerr. pick a settlement at a good watershed and show the supply overlay.

Can we make a SuppyOverlay like the way we did FoodOverlay using our system?

In [ ]:
#| export
def SupplyOverlay(settlement: int,
                  field: str = 'bishops_needed',
                  planner: SupplyPlanner = None,
                  opacity: float = 0.5,
                  **kw) -> OverlaySpec:
    """Heatmap overlay showing supply range from a settlement.
    
    Fields:
      'bishops_needed'    — green (1) → red (6+) → black (unreachable)
      'net_delivery'      — red (0) → green (high delivery)
      'max_diet'          — red (0) → green (high sustainable diet)
      'supply_efficiency' — red (0) → green (high efficiency)
      'supply_cost'       — green (cheap) → red (expensive)
    """
    STOPS = {
        'bishops_needed':    [(1, "#2ecc71"), (2, "#f1c40f"), (4, "#e67e22"),
                              (6, "#e74c3c"), (10, "#1a1a2e")],
        'net_delivery':      [(0, "#e74c3c"), (5, "#f1c40f"),
                              (10, "#2ecc71"), (20, "#27ae60")],
        'max_diet':          [(0, "#e74c3c"), (1, "#f39c12"), (2, "#f1c40f"),
                              (3, "#2ecc71"), (5, "#27ae60")],
        'supply_efficiency': [(0, "#e74c3c"), (0.5, "#f1c40f"),
                              (1, "#2ecc71"), (2, "#27ae60")],
        'supply_cost':       [(0, "#2ecc71"), (5, "#f1c40f"),
                              (10, "#e67e22"), (20, "#e74c3c")],
    }

    def lerp_color(val, color_stops):
        if np.isnan(val): return None
        if val <= color_stops[0][0]: return color_stops[0][1]
        for i in range(len(color_stops) - 1):
            v0, c0 = color_stops[i]
            v1, c1 = color_stops[i + 1]
            if val <= v1:
                t = (val - v0) / (v1 - v0) if v1 != v0 else 0
                t = max(0, min(1, t))
                r0, g0, b0 = int(c0[1:3],16), int(c0[3:5],16), int(c0[5:7],16)
                r1, g1, b1 = int(c1[1:3],16), int(c1[3:5],16), int(c1[5:7],16)
                r = int(r0 + (r1-r0)*t)
                g = int(g0 + (g1-g0)*t)
                b = int(b0 + (b1-b0)*t)
                return f"#{r:02x}{g:02x}{b:02x}"
        return color_stops[-1][1]

    def render(ctx):
        # Reuse existing planner or create one
        sp = planner or SupplyPlanner(ctx.terrain, settlement)

        # Compute if needed
        if field not in ctx.terrain.fields:
            sp.compute_supply_fields()

        data = ctx.terrain.fields[field]
        color_stops = STOPS.get(field, STOPS['bishops_needed'])
        grid = ctx.grid

        overlay = ""
        for i in range(len(data)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            color = lerp_color(data[i], color_stops)
            if color is None:
                continue

            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += (f'\t<polygon points="{pts}" '
                        f'fill="{color}" opacity="{opacity}"/>\n')

        # Settlement marker
        sc = grid.hexes[settlement].center
        r = grid.radius
        overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{r*0.6}" '
                    f'fill="none" stroke="#f1c40f" stroke-width="2.5" '
                    f'opacity="0.9"/>\n')
        overlay += (f'<text x="{sc.x}" y="{sc.y+4}" text-anchor="middle" '
                    f'font-size="{r*0.5}" fill="#f1c40f" font-weight="bold" '
                    f'font-family="sans-serif">⛺</text>\n')

        return overlay

    return OverlaySpec(f"supply_{field}", render, priority=45)


# Compare different fields
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='net_delivery'),
    terrain=myTerr,
)


# ── Pick a good settlement: fertile, low elevation, central-ish ──────────
grid = myTerr.hexGrid
elevs = myTerr.elevations
mid = grid.middle

# Get food tiers (or compute simple ones from elevation)
if 'food_yield' in myTerr.fields:
    food_tiers = myTerr.fields['food_yield']
else:
    food_tiers = np.zeros(len(elevs), dtype=int)
    for i, e in enumerate(elevs):
        if   e <= 0:    food_tiers[i] = 0
        elif e < 60:    food_tiers[i] = 7
        elif e < 120:   food_tiers[i] = 5
        elif e < 180:   food_tiers[i] = 3
        elif e < 260:   food_tiers[i] = 1
        else:           food_tiers[i] = 0
    myTerr.fields['food_yield'] = food_tiers

# Score each hex: high food tier + low elevation + close to center + many fertile neighbors
def settlement_score(idx):
    if idx in grid.invalidRegion or elevs[idx] <= 0:
        return -999
    tier = food_tiers[idx]
    if tier < 4:
        return -999
    # Count fertile neighbors
    nbs = grid.neighborsOf(idx)
    fertile_nbs = sum(1 for n in nbs
                      if n not in grid.invalidRegion
                      and elevs[n] > 0
                      and food_tiers[n] >= 3)
    # Distance from center (prefer central-ish)
    dist = abs(grid.index_to_hexposition(idx, mid))
    return tier * 3 + fertile_nbs * 2 - dist * 0.5

best_settlement = max(range(len(elevs)), key=settlement_score)
print(f"⛺ Settlement: hex {best_settlement}  "
      f"elev={elevs[best_settlement]:.0f}  "
      f"tier={food_tiers[best_settlement]}  "
      f"score={settlement_score(best_settlement):.1f}")

# ── Create planner and compute fields ────────────────────────────────────
planner = SupplyPlanner(terrain=myTerr, settlement=best_settlement)
planner.compute_supply_fields()
planner.summary()

# ── Render the base terrain + supply overlay ─────────────────────────────
myTerr.colorMap()
myTerr.hexGrid.update()

# Show bishops_needed heatmap
planner.render_supply_overlay('bishops_needed', layer_name='supply_heatmap')
show(NotStr(myTerr.hexGrid.builder.xml()))


Please make sure we are using the correct primtives. Hex Coordiantes postions can be tricky to understand. grid might have a different middle

### Battle Plans

In [ ]:
#| export
@dataclass
class BattlePlan:
    """An editable optimization result the player can tweak before committing."""
    optimizer: ColonyOptimizer
    assignments: list          # the optimizer's suggestion
    harvest_budget: float
    
    # Player overrides
    vetoed: set = field(default_factory=set)       # consumer hex_idx player said "no"
    boosted: dict = field(default_factory=dict)     # hex_idx → extra bishops
    locked: set = field(default_factory=set)        # hex_idx player said "must feed"
    
    def veto(self, hex_idx):
        """Player says: don't feed this one."""
        self.vetoed.add(hex_idx)
        return self._reoptimize()
    
    def lock(self, hex_idx):
        """Player says: this one MUST be fed, no matter what."""
        self.locked.add(hex_idx)
        return self._reoptimize()
    
    def boost(self, hex_idx, extra=1):
        """Player adds extra bishops to a route (insurance)."""
        self.boosted[hex_idx] = self.boosted.get(hex_idx, 0) + extra
        return self._reoptimize()
    
    def _reoptimize(self):
        """Re-run with player constraints applied."""
        # ... rerun optimizer with vetoed filtered out,
        #     locked as hard constraints, boosted as minimums
        return self
    
    def commit(self, board):
        """Execute the plan — place bishops, provision consumers."""
        placed = []
        for a in self.assignments:
            if not a['fed'] or a['consumer'].hex_idx in self.vetoed:
                continue
            bishops = self.optimizer.planner.create_supply_line(
                a['consumer'].hex_idx, board, n_bishops=a['bishops'])
            self.optimizer.planner.provision_consumer(
                a['consumer'].piece, target=a['consumer'].hex_idx)
            placed.extend(bishops)
        return placed


What can we do flesh this out. I do think we will comeback later when we add the other pieces - king is what must survive otherwise you loose. I am thinking of knights as fast pieces and rooks as slow tanks. but we will have to do the combat optimization part later as well (since I haven't yet built out that system). Any thoughts on this?

Can we do a similar BattlePlanOverlay as we did for SupplyOverlay. I know we would have to pass a battle plan in

In [ ]:
#| export
def BattlePlanOverlay(plan: BattlePlan,
                      show_routes: bool = True,
                      opacity: float = 0.6,
                      **kw) -> OverlaySpec:
    """Render a BattlePlan on the hex map: routes, consumer markers, bishop counts.
    
    Colors:
      ✅ fed + locked  → gold border
      ✅ fed           → green
      ❌ unfed         → red
      🚫 vetoed        → grey strikethrough
    """
    
    def render(ctx):
        grid = ctx.grid
        sp = plan.optimizer.planner
        settlement = sp.settlement
        sc = grid.hexes[settlement].center
        r = grid.radius
        
        overlay = ""
        
        for a in plan.assignments:
            c = a['consumer']
            hex_idx = c.hex_idx
            if hex_idx < 0 or hex_idx >= len(grid.hexes):
                continue
            
            tc = grid.hexes[hex_idx].center
            vetoed = hex_idx in plan.vetoed
            locked = hex_idx in plan.locked
            fed = a['fed'] and not vetoed
            
            # ── Route line from settlement to consumer ──────────
            if show_routes and not vetoed:
                # Try to get actual path for a nice hex-following line
                route = sp.plan_route(hex_idx)
                if route and route['path']:
                    path_pts = []
                    for pidx in route['path'].hexes:
                        pc = grid.hexes[pidx].center
                        path_pts.append(f"{pc.x:.0f},{pc.y:.0f}")
                    pts_str = " ".join(path_pts)
                    
                    line_color = "#2ecc71" if fed else "#e74c3c"
                    line_width = 2.5 if fed else 1.5
                    dash = "" if fed else 'stroke-dasharray="6,4"'
                    
                    overlay += (f'<polyline points="{pts_str}" '
                                f'fill="none" stroke="{line_color}" '
                                f'stroke-width="{line_width}" '
                                f'opacity="{opacity * 0.7}" '
                                f'{dash} stroke-linecap="round"/>\n')
                else:
                    # Fallback: straight line
                    line_color = "#2ecc71" if fed else "#e74c3c"
                    overlay += (f'<line x1="{sc.x:.0f}" y1="{sc.y:.0f}" '
                                f'x2="{tc.x:.0f}" y2="{tc.y:.0f}" '
                                f'stroke="{line_color}" stroke-width="1.5" '
                                f'opacity="{opacity * 0.5}" '
                                f'stroke-dasharray="4,3"/>\n')
            
            # ── Consumer hex highlight ──────────────────────────
            hex_obj = grid.hexes[hex_idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            
            if vetoed:
                fill_color = "#95a5a6"
                stroke_color = "#7f8c8d"
            elif locked:
                fill_color = "#f1c40f"
                stroke_color = "#f39c12"
            elif fed:
                fill_color = "#2ecc71"
                stroke_color = "#27ae60"
            else:
                fill_color = "#e74c3c"
                stroke_color = "#c0392b"
            
            overlay += (f'<polygon points="{pts}" '
                        f'fill="{fill_color}" opacity="{opacity * 0.4}" '
                        f'stroke="{stroke_color}" stroke-width="2" '
                        f'stroke-opacity="{opacity}"/>\n')
            
            # ── Piece icon + bishop count label ─────────────────
            piece_type = c.piece.piece_type if hasattr(c.piece, 'piece_type') else None
            icon = getattr(piece_type, 'icon', '●') if piece_type else '●'
            name = getattr(c.piece, 'name', '')[:6]
            
            # Icon
            overlay += (f'<text x="{tc.x:.0f}" y="{tc.y - r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.6}" '
                        f'fill="{stroke_color}" opacity="0.9">'
                        f'{icon}</text>\n')
            
            # Bishop count badge
            if a['bishops'] > 0 and not vetoed:
                bx = tc.x + r * 0.5
                by = tc.y - r * 0.4
                badge_color = "#2ecc71" if fed else "#e74c3c"
                overlay += (f'<circle cx="{bx:.0f}" cy="{by:.0f}" '
                            f'r="{r*0.25}" fill="{badge_color}" '
                            f'stroke="white" stroke-width="1"/>\n')
                overlay += (f'<text x="{bx:.0f}" y="{by + r*0.08:.0f}" '
                            f'text-anchor="middle" font-size="{r*0.25}" '
                            f'fill="white" font-weight="bold" '
                            f'font-family="sans-serif">'
                            f'{a["bishops"]}</text>\n')
            
            # Status label below
            status = "🚫" if vetoed else ("🔒" if locked else ("✅" if fed else "❌"))
            overlay += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.55:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.3}" '
                        f'fill="#333" font-family="sans-serif" '
                        f'font-weight="bold">{status} {name}</text>\n')
            
            # Vetoed strikethrough
            if vetoed:
                overlay += (f'<line x1="{tc.x - r*0.6:.0f}" y1="{tc.y:.0f}" '
                            f'x2="{tc.x + r*0.6:.0f}" y2="{tc.y:.0f}" '
                            f'stroke="#e74c3c" stroke-width="2.5" '
                            f'opacity="0.8"/>\n')
        
        # ── Settlement marker ───────────────────────────────────
        overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{r*0.7}" '
                    f'fill="none" stroke="#f1c40f" stroke-width="3" '
                    f'opacity="0.9"/>\n')
        overlay += (f'<text x="{sc.x}" y="{sc.y+4}" text-anchor="middle" '
                    f'font-size="{r*0.5}" fill="#f1c40f" font-weight="bold" '
                    f'font-family="sans-serif">⛺</text>\n')
        
        # ── Summary legend ──────────────────────────────────────
        fed_count = sum(1 for a in plan.assignments
                        if a['fed'] and a['consumer'].hex_idx not in plan.vetoed)
        total = len(plan.assignments)
        bishops_used = sum(a['bishops'] for a in plan.assignments
                           if a['consumer'].hex_idx not in plan.vetoed)
        
        lx, ly = r * 0.5, r * 0.5
        overlay += (f'<rect x="{lx}" y="{ly}" width="{r*8}" height="{r*1.2}" '
                    f'rx="4" fill="white" opacity="0.85" '
                    f'stroke="#ccc" stroke-width="1"/>\n')
        overlay += (f'<text x="{lx + r*0.3}" y="{ly + r*0.8}" '
                    f'font-size="{r*0.35}" font-family="sans-serif" '
                    f'fill="#333" font-weight="bold">'
                    f'🗺 {fed_count}/{total} fed · '
                    f'⛪ {bishops_used} bishops · '
                    f'🌾 {plan.harvest_budget:.0f}/turn</text>\n')
        
        return overlay
    
    return OverlaySpec("battle_plan", render, priority=50)


Can we do a BattlePieceOverlay that would show the individual piece movements that are proposed?

def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       **kw) -> OverlaySpec:
    """Render proposed piece movements from a BattlePlan.
    
    Shows simulated patrol routes for:
      - Bishops on supply runs (arrows + GIVE glyphs)
      - Consumers at their positions (PAUSE/DEFEND glyphs)
      - Optionally pawns at settlement (HARVEST/GIVE loops)
    """

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        countries = ctx.terrain.fields.get('country')
        r = grid.radius
        svg = ""

        # Collect all pieces from the plan
        pieces_to_show = []

        for a in plan.assignments:
            c = a['consumer']
            if c.hex_idx in plan.vetoed:
                continue

            # Consumer piece — show their orders (PAUSE, DEFEND, etc.)
            if show_consumers and hasattr(c, 'piece') and c.piece is not None:
                p = c.piece
                if p.location is not None and p.location >= 0:
                    pieces_to_show.append(('consumer', p, a['fed']))

        # Bishops created by the plan (if committed, they'll be on the board)
        # For pre-commit preview, we simulate from the plan's assignments
        if show_bishops:
            sp = plan.optimizer.planner
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed:
                    continue
                if a['bishops'] <= 0:
                    continue

                # Create a phantom bishop for each assigned route
                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']:
                    continue

                for bi in range(a['bishops']):
                    # Find a start hex near settlement
                    start = sp.settlement
                    nbs = list(grid.neighborsOf(start))
                    valid = [n for n in nbs
                             if n not in grid.invalidRegion and elevs[n] > 0]
                    b_start = valid[bi % len(valid)] if valid else start

                    # Build bishop patrol: out → GIVE → back → GIVE
                    path_out = route['path']
                    path_back_piece = Piece(location=c.hex_idx)
                    path_back = path_back_piece.pathfind(b_start, grid, elevs)

                    b_facing = _facing_toward(grid, b_start, c.hex_idx)
                    if path_out and path_back:
                        rules = (path_out.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value]
                                 + path_back.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value])
                    else:
                        rules = [Instruction.FORWARD.value,
                                 Instruction.GIVE.value]

                    phantom = Piece(
                        location=b_start,
                        piece_type=PieceType.BISHOP,
                        facing=b_facing,
                        instructions=InstructionList(rules, cursor=0, patrol=True),
                        owner_id=c.piece.owner_id if hasattr(c.piece, 'owner_id') else 1,
                        name=f"Supply Bishop {bi+1}",
                    )
                    # Copy flag from consumer if available
                    if hasattr(c.piece, 'flag'):
                        phantom.flag = c.piece.flag
                    FoodProfile.apply(phantom)
                    pieces_to_show.append(('bishop', phantom, True))

        # Pawns at settlement
        if show_pawns:
            sp = plan.optimizer.planner
            for pi in range(plan.optimizer.max_pawns):
                nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
                valid = [n for n in nbs
                         if n not in grid.invalidRegion and elevs[n] > 0]
                p_loc = valid[pi % len(valid)] if valid else sp.settlement

                phantom = Piece(
                    location=p_loc,
                    piece_type=PieceType.PAWN,
                    facing=random.randint(0, 5),
                    instructions=InstructionList(
                        [Instruction.HARVEST.value, Instruction.GIVE.value],
                        cursor=0, patrol=True),
                    name=f"Farmer {pi+1}",
                )
                FoodProfile.apply(phantom)
                pieces_to_show.append(('pawn', phantom, True))

        # ── Render each piece's plan ─────────────────────────────
        for role, piece, fed in pieces_to_show:
            if piece.location is None or piece.location < 0:
                continue
            if piece.location >= len(grid.hexes):
                continue

            color = "#3498db"  # bishop blue
            if role == 'consumer':
                color = "#2ecc71" if fed else "#e74c3c"
            elif role == 'pawn':
                color = "#8D6E63"

            # Use flag color if available
            if hasattr(piece, 'flag') and piece.flag:
                color = piece.flag.primary

            g = DiagramGlyphs(color=color, size=r * 0.4, opacity=opacity)

            # Simulate
            steps = piece.simulate(grid, elevs,
                                   num_turns=num_turns,
                                   countries=countries)
            if not steps and role == 'consumer':
                # Static consumer — just draw at position
                cx, cy = grid.hexes[piece.location].center.x, grid.hexes[piece.location].center.y
                svg += g.start_marker(cx, cy, r=r * 0.5)
                continue

            # Arrow style per role
            dash = f"{r*0.2:.0f},{r*0.1:.0f}" if role == 'bishop' else f"{r*0.15:.0f},{r*0.08:.0f}"
            arrow_style = StyleCSS(
                f"bp_{role}_{id(piece) % 9999}",
                stroke=color, stroke_width=max(1, r * 0.06),
                fill="none", opacity=str(opacity * 0.7),
                stroke_dasharray=dash,
            )
            grid.builder.add_style(arrow_style)

            # Start marker
            c = grid.hexes[piece.location].center
            svg += g.start_marker(c.x, c.y, r=r * 0.45)

            # Steps
            prev_idx = piece.location
            turn_ends = {}
            for i, s in enumerate(steps):
                turn_ends[s.turn] = i

            for i, step in enumerate(steps):
                idx = step.hex_idx
                if idx < 0 or idx >= len(grid.hexes):
                    prev_idx = idx
                    continue

                cx = grid.hexes[idx].center.x
                cy = grid.hexes[idx].center.y

                if step.instruction == Instruction.FORWARD and not step.blocked:
                    if 0 <= prev_idx < len(grid.hexes) and prev_idx != idx:
                        svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
                elif step.blocked:
                    svg += g.blocked_x(cx, cy)
                else:
                    svg += g.action_glyph(step.instruction, cx, cy)

                prev_idx = idx

            # Turn-end labels (merged ranges)
            for hex_idx, t0, t1 in _turn_end_labels(steps):
                if 0 <= hex_idx < len(grid.hexes):
                    cx = grid.hexes[hex_idx].center.x
                    cy = grid.hexes[hex_idx].center.y
                    comp = piece.flag.comp if hasattr(piece, 'flag') and piece.flag else "#555"
                    svg += _range_label(g, cx, cy, t0, t1, comp)

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


Can you build a nice demo using my terr to show a battle plan to secure a nearby peak and a fertile river valley?

In [ ]:
myTerr.hexGrid.builder.layers = []

In [ ]:
# ── Setup: ensure food tiers exist ──────────────────────────────────────
grid = myTerr.hexGrid
elevs = myTerr.elevations
mid = grid.middle

if 'food_yield' not in myTerr.fields:
    fy = FoodYield(myTerr, basins, n_tiers=8)
    fy.compute()

food_tiers = myTerr.fields['food_yield']

# ── 1. Find the best settlement: fertile, good neighbors ───────────────
def settlement_score(idx):
    if idx in grid.invalidRegion or elevs[idx] <= 0: return -999
    tier = int(food_tiers[idx])
    if tier < 4: return -999
    nbs = grid.neighborsOf(idx)
    fertile_nbs = sum(1 for n in nbs
                      if n not in grid.invalidRegion
                      and elevs[n] > 0 and food_tiers[n] >= 3)
    dist = abs(grid.index_to_hexposition(idx, mid))
    return tier * 3 + fertile_nbs * 2 - dist * 0.3

settlement = max(range(len(elevs)), key=settlement_score)
print(f"⛺ Settlement: hex {settlement}  "
      f"elev={elevs[settlement]:.0f}  tier={int(food_tiers[settlement])}")

# ── 2. Find a nearby peak (strategic high ground) ──────────────────────
planner = SupplyPlanner(terrain=myTerr, settlement=settlement)
planner.compute_supply_fields()
max_cost = planner.max_range()

# Dijkstra outward — find highest reachable peak within supply range
peak_target = None
peak_elev = 0
pq = [(0.0, settlement)]
visited = {}

while pq:
    cost, cur = heapq.heappop(pq)
    if cur in visited: continue
    visited[cur] = cost

    # Peak candidate: high elevation, reachable, within supply range
    if (cur != settlement
        and elevs[cur] >= 200
        and cost <= max_cost * 0.7  # not at the absolute edge of range
        and elevs[cur] > peak_elev):
        peak_target = cur
        peak_elev = elevs[cur]

    for nb in grid.neighborsOf(cur):
        if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
        new_cost = cost + _move_cost(elevs, cur, nb)
        if new_cost <= max_cost and nb not in visited:
            heapq.heappush(pq, (new_cost, nb))

print(f"🏔 Peak target: hex {peak_target}  "
      f"elev={elevs[peak_target]:.0f}  "
      f"tier={int(food_tiers[peak_target])}  "
      f"cost={visited[peak_target]:.1f}")

# ── 3. Find a fertile river valley (high food, near water) ─────────────
valley_target = None
valley_score = -999

for idx, cost in visited.items():
    if idx == settlement or cost > max_cost * 0.6:
        continue
    tier = int(food_tiers[idx])
    if tier < 5:
        continue
    # Bonus for being near water (low-elevation neighbors)
    nbs = grid.neighborsOf(idx)
    water_adj = sum(1 for n in nbs if elevs[n] <= 0)
    low_adj = sum(1 for n in nbs
                  if 0 < elevs[n] < 80 and food_tiers[n] >= 4)
    score = tier * 2 + water_adj * 3 + low_adj * 2 - cost * 0.5

    # Must be somewhat distant from settlement (not right next door)
    if cost < 3:
        continue

    if score > valley_score:
        valley_score = score
        valley_target = idx

print(f"🌾 Valley target: hex {valley_target}  "
      f"elev={elevs[valley_target]:.0f}  "
      f"tier={int(food_tiers[valley_target])}  "
      f"cost={visited[valley_target]:.1f}")

# ── 4. Create pieces at targets ─────────────────────────────────────────
# Knight on the peak (fast scout, holds high ground)
knight = Piece(
    location=peak_target,
    piece_type=PieceType.KNIGHT,
    facing=_facing_away(grid, peak_target, settlement),
    name="Sir Edmund",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(knight)

# Rook in the valley (slow tank, holds the river crossing)
rook = Piece(
    location=valley_target,
    piece_type=PieceType.ROOK,
    facing=_facing_away(grid, valley_target, settlement),
    name="Fort Riverside",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(rook)

# Queen midway between — projecting power
queen_target = planner.find_target(min_elev=80, max_tier=4,
                                    min_cost=2.0, max_cost=max_cost * 0.4)
if queen_target is None:
    queen_target = settlement  # fallback

queen = Piece(
    location=queen_target,
    piece_type=PieceType.QUEEN,
    facing=_facing_toward(grid, queen_target, peak_target),
    name="Her Majesty",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(queen)

print(f"👑 Queen: hex {queen_target}  "
      f"elev={elevs[queen_target]:.0f}  "
      f"cost={visited.get(queen_target, 0):.1f}")

# ── 5. Run the optimizer ────────────────────────────────────────────────
opt = ColonyOptimizer(planner, max_pawns=6, max_bishops=8)
consumers = opt.analyze_consumers([queen, knight, rook])
result = opt.optimize(consumers)
opt._last_result = result  # for __ft__
opt.print_plan(result)

# ── 6. Build the BattlePlan ────────────────────────────────────────────
battle_plan = BattlePlan(
    optimizer=opt,
    assignments=result['assignments'],
    harvest_budget=result['harvest_available'],
)

# ── 7. Show everything ─────────────────────────────────────────────────
print("\n🗺 Rendering battle plan...")

# Supply range + battle plan + piece movements
TerrainDisplay(
    CreamOverlay(),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    #SupplyOverlay(settlement=settlement, planner=planner, opacity=0.3),
    #BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10, show_pawns=True),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
show(myTerr.hexGrid.builder)

It didn't show any pieces

def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       **kw) -> OverlaySpec:
    """Render piece positions and planned bishop routes from a BattlePlan."""

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        r = grid.radius
        svg = ""
        sp = plan.optimizer.planner

        # ── Consumer markers ────────────────────────────────────
        if show_consumers:
            for a in plan.assignments:
                c = a['consumer']
                if c.hex_idx in plan.vetoed:
                    continue
                if c.hex_idx < 0 or c.hex_idx >= len(grid.hexes):
                    continue

                tc = grid.hexes[c.hex_idx].center
                fed = a['fed']
                color = "#2ecc71" if fed else "#e74c3c"
                name = getattr(c.piece, 'name', '')[:8]
                ptype = c.piece.piece_type.name if hasattr(c.piece, 'piece_type') else '?'
                icon = getattr(c.piece.piece_type, 'icon', '●') if hasattr(c.piece, 'piece_type') else '●'

                # Piece icon
                svg += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.7}" '
                        f'fill="{color}" opacity="{opacity}">{icon}</text>\n')

                # Name label
                svg += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.65:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.28}" '
                        f'fill="#333" font-family="sans-serif" '
                        f'font-weight="bold">{name}</text>\n')

        # ── Bishop routes (as polylines along pathfind hexes) ───
        if show_bishops:
            bishop_colors = ["#3498db", "#2980b9", "#1abc9c",
                             "#16a085", "#8e44ad", "#2c3e50"]
            bi = 0
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed:
                    continue
                if a['bishops'] <= 0:
                    continue

                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']:
                    continue

                for _ in range(a['bishops']):
                    color = bishop_colors[bi % len(bishop_colors)]
                    bi += 1

                    # Outbound path
                    path_pts = []
                    for pidx in route['path'].hexes:
                        if 0 <= pidx < len(grid.hexes):
                            pc = grid.hexes[pidx].center
                            # Slight offset per bishop so parallel routes don't overlap
                            offset = (bi % 3 - 1) * r * 0.15
                            path_pts.append(f"{pc.x + offset:.0f},{pc.y + offset:.0f}")

                    if len(path_pts) >= 2:
                        pts_str = " ".join(path_pts)
                        svg += (f'<polyline points="{pts_str}" '
                                f'fill="none" stroke="{color}" '
                                f'stroke-width="{r * 0.08:.1f}" '
                                f'opacity="{opacity * 0.6}" '
                                f'stroke-dasharray="{r*0.3:.0f},{r*0.15:.0f}" '
                                f'stroke-linecap="round" '
                                f'marker-end="url(#arrow_{color[1:]})"/>\n')

                    # Bishop start marker (small circle at settlement)
                    sc = grid.hexes[sp.settlement].center
                    svg += (f'<circle cx="{sc.x + (bi%3-1)*r*0.3:.0f}" '
                            f'cy="{sc.y + (bi//3)*r*0.3:.0f}" '
                            f'r="{r*0.2}" fill="{color}" '
                            f'opacity="{opacity * 0.8}" '
                            f'stroke="white" stroke-width="1"/>\n')

                    # GIVE marker at destination
                    tc = grid.hexes[c.hex_idx].center
                    svg += (f'<text x="{tc.x + r*0.4:.0f}" '
                            f'y="{tc.y - r*0.3:.0f}" '
                            f'text-anchor="middle" font-size="{r*0.3}" '
                            f'fill="{color}" opacity="{opacity}">🌾</text>\n')

        # ── Pawn cluster at settlement ──────────────────────────
        if show_pawns:
            sc = grid.hexes[sp.settlement].center
            nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
            valid = [n for n in nbs
                     if n not in grid.invalidRegion and elevs[n] > 0]
            for pi in range(min(plan.optimizer.max_pawns, len(valid))):
                pc = grid.hexes[valid[pi]].center
                svg += (f'<text x="{pc.x:.0f}" y="{pc.y + r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.5}" '
                        f'fill="#8D6E63" opacity="{opacity * 0.7}">♟</text>\n')
                svg += (f'<text x="{pc.x:.0f}" y="{pc.y + r*0.55:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.2}" '
                        f'fill="#6D4C41" font-family="sans-serif">'
                        f'🌾⇄</text>\n')

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


In [ ]:
# Check what DiagramGlyphs can do
print([m for m in dir(DiagramGlyphs) if not m.startswith('_')])

# Check if Piece has simulate
print(hasattr(Piece, 'simulate'))

# Check if grid has arrow
print(hasattr(grid, 'arrow'))

# Check helpers
print('_turn_end_labels' in dir())
print('_range_label' in dir())


Thoughts?

Yes this version did draw.

In [ ]:
#| export
def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       scale: float = 1.8,
                       **kw) -> OverlaySpec:
    """Render piece positions and planned bishop routes from a BattlePlan.
    
    Args:
        scale: Multiplier for all text/glyph sizes. Increase if text is too small.
    """

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        countries = ctx.terrain.fields.get('country')
        r = grid.radius
        s = r * scale        # scaled base unit
        svg = ""
        sp = plan.optimizer.planner

        pieces_to_show = []

        # ── Consumer pieces ─────────────────────────────────────
        if show_consumers:
            for a in plan.assignments:
                c = a['consumer']
                if c.hex_idx in plan.vetoed: continue
                if c.hex_idx < 0 or c.hex_idx >= len(grid.hexes): continue
                if hasattr(c, 'piece') and c.piece is not None:
                    pieces_to_show.append(('consumer', c.piece, a['fed']))

        # ── Phantom bishops from plan ───────────────────────────
        if show_bishops:
            bi = 0
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed: continue
                if a['bishops'] <= 0: continue

                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']: continue

                nbs = list(grid.neighborsOf(sp.settlement))
                valid = [n for n in nbs
                         if n not in grid.invalidRegion and elevs[n] > 0]

                for _ in range(a['bishops']):
                    b_start = valid[bi % len(valid)] if valid else sp.settlement
                    bi += 1
                    b_facing = _facing_toward(grid, b_start, c.hex_idx)

                    path_back = Piece(location=c.hex_idx).pathfind(
                        b_start, grid, elevs)

                    if route['path'] and path_back:
                        rules = (route['path'].to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value]
                                 + path_back.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value])
                    else:
                        rules = [Instruction.FORWARD.value,
                                 Instruction.GIVE.value]

                    phantom = Piece(
                        location=b_start,
                        piece_type=PieceType.BISHOP,
                        facing=b_facing,
                        instructions=InstructionList(rules, cursor=0, patrol=True),
                        name=f"Bishop {bi}",
                    )
                    FoodProfile.apply(phantom)
                    pieces_to_show.append(('bishop', phantom, True))

        # ── Pawn phantoms at settlement ─────────────────────────
        if show_pawns:
            nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
            valid = [n for n in nbs
                     if n not in grid.invalidRegion and elevs[n] > 0]
            for pi in range(min(plan.optimizer.max_pawns, len(valid))):
                phantom = Piece(
                    location=valid[pi],
                    piece_type=PieceType.PAWN,
                    facing=random.randint(0, 5),
                    instructions=InstructionList(
                        [Instruction.HARVEST.value, Instruction.GIVE.value],
                        cursor=0, patrol=True),
                    name=f"Farmer {pi+1}",
                )
                FoodProfile.apply(phantom)
                pieces_to_show.append(('pawn', phantom, True))

        # ── Render each piece ───────────────────────────────────
        bishop_colors = ["#3498db", "#2980b9", "#1abc9c",
                         "#16a085", "#8e44ad", "#2c3e50"]
        b_idx = 0

        for role, piece, fed in pieces_to_show:
            if piece.location is None or piece.location < 0: continue
            if piece.location >= len(grid.hexes): continue

            # Pick color by role
            if role == 'bishop':
                color = bishop_colors[b_idx % len(bishop_colors)]
                b_idx += 1
            elif role == 'consumer':
                color = "#2ecc71" if fed else "#e74c3c"
            else:
                color = "#8D6E63"

            g = DiagramGlyphs(color=color, size=s * 0.4, opacity=opacity)

            # Start marker
            hc = grid.hexes[piece.location].center
            svg += g.start_marker(hc.x, hc.y, r=s * 0.45)

            # Simulate movement
            steps = piece.simulate(grid, elevs,
                                   num_turns=num_turns,
                                   countries=countries)

            if not steps:
                # Static piece — icon + name only
                name = getattr(piece, 'name', '')[:8]
                icon = getattr(piece.piece_type, 'icon', '●')
                svg += (f'<text x="{hc.x:.0f}" y="{hc.y + s*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{s*0.7:.1f}" '
                        f'fill="{color}" opacity="{opacity}">{icon}</text>\n')
                svg += (f'<text x="{hc.x:.0f}" y="{hc.y + s*0.6:.0f}" '
                        f'text-anchor="middle" font-size="{s*0.28:.1f}" '
                        f'fill="#222" font-family="sans-serif" '
                        f'font-weight="bold">{name}</text>\n')
                continue

            # Draw simulated steps
            prev_idx = piece.location
            for step in steps:
                idx = step.hex_idx
                if idx < 0 or idx >= len(grid.hexes):
                    prev_idx = idx
                    continue

                sc = grid.hexes[idx].center

                if step.instruction == Instruction.FORWARD and not step.blocked:
                    if 0 <= prev_idx < len(grid.hexes) and prev_idx != idx:
                        p1 = grid.hexes[prev_idx].center
                        svg += (f'<line x1="{p1.x:.0f}" y1="{p1.y:.0f}" '
                                f'x2="{sc.x:.0f}" y2="{sc.y:.0f}" '
                                f'stroke="{color}" '
                                f'stroke-width="{s*0.07:.1f}" '
                                f'opacity="{opacity * 0.6}" '
                                f'stroke-dasharray="{s*0.2:.0f},{s*0.1:.0f}" '
                                f'stroke-linecap="round"/>\n')
                elif step.blocked:
                    svg += g.blocked_x(sc.x, sc.y)
                else:
                    svg += g.action_glyph(step.instruction, sc.x, sc.y)

                prev_idx = idx

            # Name label + turn at final position
            if steps:
                last = steps[-1]
                if 0 <= last.hex_idx < len(grid.hexes):
                    fc = grid.hexes[last.hex_idx].center
                    svg += g.step_label(fc.x, fc.y + s * 0.4,
                                        f"T{last.turn}")
                    # Name below turn label
                    name = getattr(piece, 'name', '')[:8]
                    svg += (f'<text x="{fc.x:.0f}" y="{fc.y + s*0.65:.0f}" '
                            f'text-anchor="middle" font-size="{s*0.28:.1f}" '
                            f'fill="#222" font-family="sans-serif" '
                            f'font-weight="bold">{name}</text>\n')

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    CreamOverlay(),
    #SupplyOverlay(settlement=settlement, planner=planner, opacity=0.3),
    BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10, show_pawns=True, scale=2.5),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


fix please

please write the full updated version with the scale parameter?